# GPPM PAAG/CSM — v10: Protocol 2 (confirmatory, 20 seeds, localisation, PadChest external evaluation)

This notebook runs **Protocol 2**. The published Protocol-1 results are reproduced by `GPPM_PAAG_CSM_v5.ipynb`. Run the shared-definition cells (S), then P1–P6 once per session, then P7 to train the seed you choose, and P8–P9 for the pre-registered analysis.


# Protocol 2 — a fair, high-powered test of PAAG and CSM

**Hypotheses are unchanged from the manuscript:** (H1) pathology-aware attention improves geriatric pneumonia
discrimination; (H2) suppressing pneumonia evidence that is explained by comorbidity (above all cardiomegaly)
reduces comorbidity-driven false positives and improves discrimination; (H3) the modules change *where* the model
looks. Protocol 1 (the published runs, notebook v5) could not test H1–H3 fairly. Protocol 2 removes those
obstacles, gives every variant a stronger shared training recipe, and is built to run one seed per session for
as many seeds as you can afford.

| Protocol-1 finding / limitation (manuscript) | Protocol-2 change | Why it serves the hypotheses |
|---|---|---|
| PAAG was a single scalar on the pooled vector: it could not attend to any region | **Spatial PAAG**: a 1×1-conv attention map over the 10×10 feature grid, used for a pneumonia-specific attended pooling, `F·(1+a)`, initialised near identity | Makes H1/H3 testable: the module can now actually select lung regions; its map is inspectable |
| CSM output was unbounded (could *raise* the pneumonia logit) and its gradient bent the comorbidity heads toward the pneumonia loss | **Bounded CSM**: `s = softplus(MLP(σ(ℓ_com)))` ≥ 0, with stop-gradient on the comorbidity evidence | CSM can only *suppress*, as designed; comorbidity heads learn only from comorbidity labels (also a candidate cause of the CSM localization drift) |
| Single-label groups: pneumonia images were trained as "no cardiomegaly", so CSM never saw co-occurrence | **Partial-label targets** from the priority rule (lower-priority groups = unknown, masked in the loss) + optional **true multi-labels** from NIH `Data_Entry_2017` and CheXpert/VisualCheXbert CSVs | Removes false "absent" labels; with the CSVs, CSM finally sees pneumonia + cardiomegaly together (manuscript Recommendation 1) |
| Checkpoint = lowest class-weighted val loss (seed-42 base kept at epoch 3 while val AUC rose to 0.829) | Checkpoint = best **validation ROC-AUC of the EMA weights**, fixed epoch budget with cosine schedule | Selects on the pre-specified endpoint; EMA + fixed budget cut the seed-to-seed variance that made Protocol 1 under-powered |
| Imbalance corrected twice (sampler + pos_weight) → Brier 0.35, thresholds ≈ 0.9 | **Single correction** (pos_weight, natural-prevalence batches); Platt scaling fitted on validation and stored with every run | Calibrated outputs; every image seen once per epoch |
| 224 px; horizontal flip only | **320 px**, mild affine + resized-crop + contrast jitter, **no horizontal flip** (cardiac laterality matters for cardiomegaly) | Higher resolution helps small opacities; flips create anatomically impossible hearts |
| CLAHE recomputed on full-size images every epoch (≈ 6 h/run) | **One-off CLAHE cache** at training resolution | Several-fold faster epochs → more seeds per GPU-hour |
| 5 seeds; MDD ≈ 0.012–0.018 ROC-AUC | **Pre-registered seed pool** (40 seeds, target 20 complete seeds), one (variant, seed) per session, **resumable** across sessions, config hash enforced | Lets you add seeds safely over many sessions without hyperparameter drift or seed cherry-picking |
| Shared test set; image-level tests ignore patient clustering | Primary analysis = **hierarchical bootstrap over seeds and patients**, Holm across the three primary contrasts | One analysis that accounts for both training randomness and patient sampling |
| Central-box Grad-CAM reference | **Anatomical reference** (lungs, heart) from the ChestX-Det PSPNet in `torchxrayvision`, with the central box kept as a fallback | Measures attention on lungs (pneumonia cases) and on the heart (cardiomegaly cases) — the mechanism behind 74% of false positives |
| Pooled AUC inflated by source differences | **Within-CheXpert AUC** reported as a key secondary endpoint for every run | Guards against shortcut learning between datasets |

### Pre-registered analysis plan (write it down before training seed 1)
* **Primary endpoint:** test-set ROC-AUC. **Primary contrasts:** PAAG − base, CSM − base, Full − base. Hierarchical
  (seed × patient) bootstrap, 95% CI, Holm-adjusted p.
* **Key secondary:** within-CheXpert ROC-AUC; cardiomegaly false-positive rate at a validation-fixed specificity of
  0.80; Brier score after Platt scaling; lung- and heart-attention ratios.
* **Sample size:** the **first 20 seeds of the registry**, each with all four variants. Interim looks show
  estimates only; p-values are released when all 20 are complete (no optional stopping, no seed selection).
* **Development vs confirmation:** tune anything you want on the **validation** set during development runs
  (`STAGE = 'dev'`). Once `STAGE = 'confirm'`, the configuration is frozen by its hash and test results are read
  only through the pre-registered analysis.

### How to run (one session = one or more (variant, seed) jobs)
1. Run the shared-definition cells, then P1–P6 (they only define things and build caches/labels).
2. In **P7**, set `SESSION_SEED` (from the registry) and optionally `SESSION_VARIANTS`, then run it.
   Interrupted runs resume automatically from the last finished epoch.
3. Run **P8** and **P9** at any time; they report only what is complete.

Protocol-1 (published) results remain reproducible with notebook v5. Protocol-2 results are a new experiment and
must be reported as such.


## S. Shared definitions (data paths, filename parsing, folder loader, Grad-CAM/IoU helpers)
Identical to v5. Only `BASE_DIR` may need editing.

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (confusion_matrix,
                             precision_recall_curve, roc_curve, auc,
                             precision_score, recall_score, f1_score, roc_auc_score)
import warnings
import cv2
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.patches as patches
from sklearn.calibration import calibration_curve
from scipy import stats as scipy_stats
import json
import ssl
import random

warnings.filterwarnings('ignore')

In [ ]:
RANDOM_STATE = 42
BATCH_SIZE = 190
LEARNING_RATE = 1e-4
NUM_EPOCHS = 40
ABLATION_NUM_EPOCHS = 15
NUM_WORKERS = min(8, os.cpu_count() or 4)

EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 0.0

MAX_POS_WEIGHT = 20.0

WEIGHT_DECAY = 1e-5
USE_DIFFERENTIAL_LR = True
BACKBONE_LR_MULTIPLIER = 0.1

LR_SCHEDULER_ENABLED = True
LR_SCHEDULER_FACTOR = 0.5
LR_SCHEDULER_PATIENCE = 3
LR_SCHEDULER_MIN_LR = 1e-6

IMG_RESIZE_SIZE = 256
IMG_CROP_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)

ATTENTION_HIDDEN_DIM = 512
SUPPRESSOR_HIDDEN_DIM = 64

PCI_SAMPLER_INITIAL_RATIO = 0.5
PCI_SAMPLER_RATIO_INCREMENT = 0.02
PCI_SAMPLER_MAX_RATIO = 0.7

STAT_ALPHA = 0.05
STAT_N_BOOTSTRAP = 2000

IOU_THRESHOLDS = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7)
IOU_NUM_SAMPLES = 100
IOU_NUM_VISUALIZE = 6

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f"RANDOM_STATE={RANDOM_STATE}, BATCH_SIZE={BATCH_SIZE}, LEARNING_RATE={LEARNING_RATE}, NUM_EPOCHS={NUM_EPOCHS}, NUM_WORKERS={NUM_WORKERS}, EARLY_STOPPING_PATIENCE={EARLY_STOPPING_PATIENCE}, WEIGHT_DECAY={WEIGHT_DECAY}, USE_DIFFERENTIAL_LR={USE_DIFFERENTIAL_LR}, LR_SCHEDULER_ENABLED={LR_SCHEDULER_ENABLED}, MAX_POS_WEIGHT={MAX_POS_WEIGHT}")


# ----------------------------------------------------------------------------
# Training-design switches added in v5.
# The defaults reproduce the 20 published runs exactly; change them only for
# sensitivity experiments. Any non-default setting writes to a separate, tagged
# results folder (see run_tag()), so published checkpoints are never overwritten.
# ----------------------------------------------------------------------------
# How pneumonia class imbalance is corrected during training:
#   'both'            pneumonia oversampling (EnhancedPCISampler, 50%->70%) AND pos_weight in the loss  [published]
#   'pos_weight_only' natural-prevalence shuffled batches + pos_weight (single correction)
#   'sampler_only'    oversampling only, unweighted pneumonia loss (single correction)
#   'none'            neither (reference run)
IMBALANCE_STRATEGY = 'both'
# Which validation quantity selects the retained checkpoint and drives early stopping:
#   'val_loss' (class-weighted validation loss, minimised) [published]  or  'val_auc' (validation ROC-AUC, maximised)
CHECKPOINT_METRIC = 'val_loss'

PUBLISHED_CONFIG = {'IMBALANCE_STRATEGY': 'both', 'CHECKPOINT_METRIC': 'val_loss'}


def run_tag():
    """'' for the published configuration, otherwise a suffix that keeps sensitivity runs in their own folders."""
    if IMBALANCE_STRATEGY == PUBLISHED_CONFIG['IMBALANCE_STRATEGY'] and CHECKPOINT_METRIC == PUBLISHED_CONFIG['CHECKPOINT_METRIC']:
        return ''
    return f"__imb-{IMBALANCE_STRATEGY}__ckpt-{CHECKPOINT_METRIC}"


def current_run_config(**extra):
    """Every hyperparameter that can differ between runs, so each checkpoint folder records how it was made."""
    cfg = {
        'RANDOM_STATE': RANDOM_STATE, 'BATCH_SIZE': BATCH_SIZE, 'LEARNING_RATE': LEARNING_RATE,
        'NUM_EPOCHS': NUM_EPOCHS, 'EARLY_STOPPING_PATIENCE': EARLY_STOPPING_PATIENCE,
        'EARLY_STOPPING_MIN_DELTA': EARLY_STOPPING_MIN_DELTA, 'MAX_POS_WEIGHT': MAX_POS_WEIGHT,
        'WEIGHT_DECAY': WEIGHT_DECAY, 'USE_DIFFERENTIAL_LR': USE_DIFFERENTIAL_LR,
        'BACKBONE_LR_MULTIPLIER': BACKBONE_LR_MULTIPLIER, 'LR_SCHEDULER_ENABLED': LR_SCHEDULER_ENABLED,
        'LR_SCHEDULER_FACTOR': LR_SCHEDULER_FACTOR, 'LR_SCHEDULER_PATIENCE': LR_SCHEDULER_PATIENCE,
        'LR_SCHEDULER_MIN_LR': LR_SCHEDULER_MIN_LR, 'IMG_RESIZE_SIZE': IMG_RESIZE_SIZE, 'IMG_CROP_SIZE': IMG_CROP_SIZE,
        'CLAHE_CLIP_LIMIT': CLAHE_CLIP_LIMIT, 'CLAHE_TILE_GRID_SIZE': list(CLAHE_TILE_GRID_SIZE),
        'ATTENTION_HIDDEN_DIM': ATTENTION_HIDDEN_DIM, 'SUPPRESSOR_HIDDEN_DIM': SUPPRESSOR_HIDDEN_DIM,
        'PCI_SAMPLER_INITIAL_RATIO': PCI_SAMPLER_INITIAL_RATIO, 'PCI_SAMPLER_RATIO_INCREMENT': PCI_SAMPLER_RATIO_INCREMENT,
        'PCI_SAMPLER_MAX_RATIO': PCI_SAMPLER_MAX_RATIO, 'IMBALANCE_STRATEGY': IMBALANCE_STRATEGY,
        'CHECKPOINT_METRIC': CHECKPOINT_METRIC, 'AMP': bool(torch.cuda.is_available()),
        'CUDNN_BENCHMARK': bool(torch.cuda.is_available()),
        'torch_version': torch.__version__,
    }
    cfg.update(extra)
    return cfg


def save_run_config(save_dir, **extra):
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, 'run_config.json'), 'w') as fh:
        json.dump(current_run_config(**extra), fh, indent=2)


print(f"IMBALANCE_STRATEGY={IMBALANCE_STRATEGY}, CHECKPOINT_METRIC={CHECKPOINT_METRIC}, run_tag='{run_tag()}'")


In [ ]:
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

In [ ]:
BASE_DIR = "/notebooks"

TRAIN_DIR = os.path.join(BASE_DIR, "Training")
VAL_DIR = os.path.join(BASE_DIR, "Validation")
TEST_DIR = os.path.join(BASE_DIR, "Testing")

PRIMARY_TARGET = "Pneumonia_Infectious"
COMORBIDITY_GROUPS = [
    "Cardiomegaly",
    "Focal_Lesion_Chronic_Change",
    "No_Finding",
    "Pleural_Extrapulmonary_Volume_Disease",
    "Weakly_Supervised_Uncertain",
    PRIMARY_TARGET,
]
UNCERTAIN_GROUP = "Weakly_Supervised_Uncertain"

IMAGE_DIRS = [d for d in [TRAIN_DIR, VAL_DIR, TEST_DIR] if os.path.isdir(d)]

print("=" * 78)
print("Detected Paperspace/Gradient configuration (no CSVs -- folder + filename labels):")
print(f"  BASE_DIR   = {BASE_DIR}")
print(f"  TRAIN_DIR  = {TRAIN_DIR}  (exists: {os.path.isdir(TRAIN_DIR)})")
print(f"  VAL_DIR    = {VAL_DIR}  (exists: {os.path.isdir(VAL_DIR)})")
print(f"  TEST_DIR   = {TEST_DIR}  (exists: {os.path.isdir(TEST_DIR)})")
print(f"  PRIMARY_TARGET (Pneumonia)  = {PRIMARY_TARGET}")
print(f"  COMORBIDITY_GROUPS ({len(COMORBIDITY_GROUPS)}) = {COMORBIDITY_GROUPS}")
print("  All images are treated as already belonging to the geriatric cohort; no age-based filtering is applied.")
for _split_name, _split_dir in [("Training", TRAIN_DIR), ("Validation", VAL_DIR), ("Testing", TEST_DIR)]:
    if os.path.isdir(_split_dir):
        _found = [g for g in COMORBIDITY_GROUPS if os.path.isdir(os.path.join(_split_dir, g))]
        _missing = [g for g in COMORBIDITY_GROUPS if g not in _found]
        print(f"  {_split_name}: found group subfolders {_found}"
              + (f"  !! MISSING: {_missing}" if _missing else ""))
print("If any of these are wrong for your setup, edit the variables above directly.")
print("=" * 78)

RESULTS_ROOT = os.path.join(BASE_DIR, "results_PAAG_CSM")

train_visualizations   = os.path.join(RESULTS_ROOT, "TrainingViz")
test_visualizations    = os.path.join(RESULTS_ROOT, "TestViz")
model_save_path        = os.path.join(RESULTS_ROOT, "BestModel")
ablation_studies_path  = os.path.join(RESULTS_ROOT, "AblationStudies")
cohort_description_dir = os.path.join(RESULTS_ROOT, "CohortDescription")

for _d in [RESULTS_ROOT, train_visualizations, test_visualizations, model_save_path,
           ablation_studies_path, cohort_description_dir]:
    os.makedirs(_d, exist_ok=True)

print(f"All results will be saved under: {RESULTS_ROOT}")


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
class CLAHETransform:
    def __init__(self, clip_limit=None, tile_grid_size=None):
        self.clip_limit = clip_limit if clip_limit is not None else CLAHE_CLIP_LIMIT
        self.tile_grid_size = tile_grid_size if tile_grid_size is not None else CLAHE_TILE_GRID_SIZE
        
        self.clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)

    def __call__(self, img):
        img_np = np.array(img.convert('L'))
        
        clahe_img = self.clahe.apply(img_np)
        
        clahe_pil = Image.fromarray(clahe_img).convert('RGB')
        
        return clahe_pil

class GeriatricMedicalImageDatasetEnhanced(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, multi_label=False, label_columns=None):
        self.dataframe = dataframe
        self.transform = transform
        self.multi_label = multi_label
        self.label_columns = label_columns
        self._has_full_path = 'Full Path' in dataframe.columns if dataframe is not None else False

        if isinstance(image_dir, (list, tuple)):
            self._search_dirs = [d for d in image_dir if d]
            self.image_dir = self._search_dirs[0] if self._search_dirs else None
        else:
            self.image_dir = image_dir
            self._search_dirs = [image_dir] if image_dir else []

        self._path_index = {}
        if not self._has_full_path:
            for d in self._search_dirs:
                if d and os.path.isdir(d):
                    for root, _dirs, files in os.walk(d):
                        for fname in files:
                            self._path_index.setdefault(fname, os.path.join(root, fname))

    def resolve_image_path(self, img_name):
        path = self._path_index.get(img_name)
        if path is not None:
            return path
        for d in self._search_dirs:
            candidate = os.path.join(d, img_name)
            if os.path.exists(candidate):
                return candidate
        base = self._search_dirs[0] if self._search_dirs else ''
        return os.path.join(base, img_name)

    def get_image_path(self, idx):
        row = self.dataframe.iloc[idx]
        if self._has_full_path:
            return row['Full Path']
        return self.resolve_image_path(row['Image Index'])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.get_image_path(idx)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)

        if self.multi_label:
            labels = self.dataframe.iloc[idx][self.label_columns].values.astype(np.float32)
        else:
            labels = self.dataframe.iloc[idx]['label_encoded']

        pneumonia_label = self.dataframe.iloc[idx]['Pneumonia'] if 'Pneumonia' in self.dataframe.columns else 0
        sample_id = self.dataframe.iloc[idx].get('Patient ID', idx)
        
        return image, torch.tensor(labels), torch.tensor(pneumonia_label, dtype=torch.float32), str(sample_id)
        
class EnhancedPCISampler:
    def __init__(self, dataset, pneumonia_indices, non_pneumonia_indices, 
                 pneumonia_ratio=None, dynamic_ratio=True):
        self.dataset = dataset
        self.pneumonia_indices = pneumonia_indices
        self.non_pneumonia_indices = non_pneumonia_indices
        self.pneumonia_ratio = pneumonia_ratio if pneumonia_ratio is not None else PCI_SAMPLER_INITIAL_RATIO
        self.dynamic_ratio = dynamic_ratio
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        if self.dynamic_ratio:
            current_ratio = min(self.pneumonia_ratio + (self.epoch * PCI_SAMPLER_RATIO_INCREMENT), PCI_SAMPLER_MAX_RATIO)
        else:
            current_ratio = self.pneumonia_ratio

        n_total = len(self.dataset)
        n_pneumonia = int(n_total * current_ratio)
        n_non_pneumonia = n_total - n_pneumonia

        pneumonia_sample = np.random.choice(self.pneumonia_indices, n_pneumonia,
                                           replace=len(self.pneumonia_indices) < n_pneumonia)
        non_pneumonia_sample = np.random.choice(self.non_pneumonia_indices, n_non_pneumonia,
                                               replace=len(self.non_pneumonia_indices) < n_non_pneumonia)

        indices = np.concatenate([pneumonia_sample, non_pneumonia_sample])
        np.random.shuffle(indices)

        return iter(indices)

    def __len__(self):
        return len(self.dataset)

class AttentionHead(nn.Module):
    def __init__(self, in_features, hidden_features):
        super(AttentionHead, self).__init__()
        self.attention_head = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.attention_head(x)

class NoOpGate(nn.Module):
    def forward(self, x):
        return torch.ones(x.size(0), 1, device=x.device, dtype=x.dtype)

class ComorbiditySuppressor(nn.Module):
    def __init__(self, num_comorbidity_classes, hidden_features=64):
        super(ComorbiditySuppressor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_comorbidity_classes, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, 1)
        )

    def forward(self, comorbidity_logits):
        return self.net(comorbidity_logits)

class NoOpSuppressor(nn.Module):
    def forward(self, comorbidity_logits):
        return torch.zeros(comorbidity_logits.size(0), 1, device=comorbidity_logits.device, dtype=comorbidity_logits.dtype)

class EnhancedPneumoniaDenseNet(nn.Module):
    def __init__(self, num_classes, comorbidity_classes, pneumonia_index, multi_label=False, use_pretrained=True):
        super(EnhancedPneumoniaDenseNet, self).__init__()
        self.multi_label = multi_label
        self.pneumonia_index = pneumonia_index
        self.comorbidity_classes = comorbidity_classes
        
        densenet = models.densenet121(pretrained=use_pretrained)
        self.features = densenet.features
        
        num_features = densenet.classifier.in_features
        self.classifier = nn.Sequential(
            nn.Linear(num_features, num_classes)
        )
        
        self.attention_head = AttentionHead(num_features, ATTENTION_HIDDEN_DIM)
        self.comorbidity_suppressor = ComorbiditySuppressor(len(comorbidity_classes), SUPPRESSOR_HIDDEN_DIM)
        
    def forward(self, x):
        features = self.features(x)
        out = F.relu(features, inplace=False)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        
        attention_weights = self.attention_head(out)
        attended_features = out * attention_weights
        
        logits = self.classifier(attended_features)
        
        comorbidity_logits = logits[:, self.comorbidity_classes]
        raw_pneumonia_logits = logits[:, self.pneumonia_index].unsqueeze(1)
        suppression = self.comorbidity_suppressor(comorbidity_logits)
        pneumonia_logits = raw_pneumonia_logits - suppression

        return logits, attention_weights, pneumonia_logits, comorbidity_logits

In [ ]:
import re as _re

_CHEXPERT_RE = _re.compile(
    r'^CheXpert_(?P<pid>patient\d+)_(?P<age>\d{1,3})_(?P<sex>[A-Za-z]+)_(?P<view>[A-Za-z]+)(?:_|\.)',
    _re.IGNORECASE)
_NIH_RE = _re.compile(
    r'^ChestX-ray8_(?P<pid>\d+)_(?P<age>\d{1,3})_(?P<sex>[A-Za-z]+)_(?P<view>[A-Za-z]+)_(?P<img>\d{8})_\d{3}',
    _re.IGNORECASE)

_SEX_MAP = {'female': 'Female', 'f': 'Female', 'male': 'Male', 'm': 'Male'}
_VIEW_MAP = {'ap': 'AP', 'pa': 'PA'}


def _norm_sex(token):
    return _SEX_MAP.get(str(token).strip().lower(), 'Unknown')


def _norm_view(token):
    return _VIEW_MAP.get(str(token).strip().lower(), 'Unknown')


def _norm_age(token):
    try:
        age = int(token)
    except (TypeError, ValueError):
        return None
    return age if 0 <= age <= 120 else None


def parse_filename_metadata(filename: str) -> dict:
    """Return {'source','patient_id','age','sex','view','parse_mode'} for one image filename."""
    name = os.path.basename(str(filename))

    m = _CHEXPERT_RE.match(name)
    if m:
        return {'source': 'CheXpert', 'patient_id': f"CXP_{m.group('pid').lower()}",
                'age': _norm_age(m.group('age')), 'sex': _norm_sex(m.group('sex')),
                'view': _norm_view(m.group('view')), 'parse_mode': 'chexpert_pattern'}

    m = _NIH_RE.match(name)
    if m:
        return {'source': 'ChestX-ray8', 'patient_id': f"NIH_{int(m.group('pid')):08d}",
                'age': _norm_age(m.group('age')), 'sex': _norm_sex(m.group('sex')),
                'view': _norm_view(m.group('view')), 'parse_mode': 'nih_pattern'}

    # Fallback: exact-token matching only (never substring matching).
    stem = os.path.splitext(name)[0]
    tokens = [t for t in _re.split(r'[_\s]+', stem) if t]
    lower = [t.lower() for t in tokens]
    sex = next((_SEX_MAP[t] for t in lower if t in _SEX_MAP), 'Unknown')
    view = next((_VIEW_MAP[t] for t in lower if t in _VIEW_MAP), 'Unknown')
    pid = next((t for t in tokens if t.lower().startswith('patient')), None)
    if pid is None:
        pid = next((t for t in tokens if t.isdigit() and len(t) >= 5), 'Unknown')
    # age = a 1-3 digit token immediately before the sex token, if there is one
    age = None
    sex_pos = next((k for k, t in enumerate(lower) if t in _SEX_MAP), None)
    if sex_pos is not None and sex_pos > 0:
        age = _norm_age(tokens[sex_pos - 1]) if tokens[sex_pos - 1].isdigit() else None
    return {'source': 'Unknown', 'patient_id': pid, 'age': age, 'sex': sex, 'view': view,
            'parse_mode': 'fallback_tokens'}


# Backward-compatible wrappers
def extract_image_view(filename: str) -> str:
    return parse_filename_metadata(filename)['view']


def extract_patient_age(filename: str):
    return parse_filename_metadata(filename)['age']


def extract_patient_gender(filename: str) -> str:
    return parse_filename_metadata(filename)['sex']


def extract_patient_id(filename: str) -> str:
    return parse_filename_metadata(filename)['patient_id']


def _self_test_filename_parsers():
    cases = {
        'CheXpert_patient00143_84_Female_AP_view1_frontal.jpg': ('CXP_patient00143', 84, 'Female', 'AP'),
        'CheXpert_patient00426_70_Male_PA_view1_frontal.jpg':   ('CXP_patient00426', 70, 'Male', 'PA'),
        'ChestX-ray8_16036_71_M_AP_00016036_002.png':           ('NIH_00016036', 71, 'Male', 'AP'),
        'ChestX-ray8_99_70_F_PA_00000099_001.png':              ('NIH_00000099', 70, 'Female', 'PA'),
    }
    for fname, (pid, age, sex, view) in cases.items():
        got = parse_filename_metadata(fname)
        assert (got['patient_id'], got['age'], got['sex'], got['view']) == (pid, age, sex, view), (fname, got)
    print("Filename parser self-test passed (CheXpert and ChestX-ray8 formats; 'Female' no longer parsed as 'Male').")


_self_test_filename_parsers()


def effective_pos_weight(pos_weight_value):
    """Pneumonia pos_weight actually used in the loss under IMBALANCE_STRATEGY."""
    if IMBALANCE_STRATEGY in ('both', 'pos_weight_only'):
        return pos_weight_value
    return 1.0


def make_train_loader(train_dataset, df_train, batch_size, num_workers):
    """Training DataLoader under IMBALANCE_STRATEGY ('both'/'sampler_only' -> EnhancedPCISampler,
    'pos_weight_only'/'none' -> ordinary shuffled batches at natural prevalence)."""
    common = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=torch.cuda.is_available(),
                  persistent_workers=num_workers > 0)
    if IMBALANCE_STRATEGY in ('both', 'sampler_only'):
        pneumonia_indices = df_train[df_train['Pneumonia'] == 1].index.tolist()
        non_pneumonia_indices = df_train[df_train['Pneumonia'] == 0].index.tolist()
        sampler = EnhancedPCISampler(train_dataset, pneumonia_indices, non_pneumonia_indices)
        return DataLoader(train_dataset, sampler=sampler, **common)
    return DataLoader(train_dataset, shuffle=True, **common)


def calculate_full_metrics(y_true, y_pred_proba, optimal_threshold):
    y_true = np.asarray(y_true)
    y_pred_proba = np.asarray(y_pred_proba)
    if len(np.unique(y_true)) < 2:
        return {
            'ROC-AUC': 0.0, 'Optimal F1': 0.0, 'Precision': 0.0,
            'Recall': 0.0, 'Specificity': 0.0, 'Brier Score': 0.0,
            'Optimal Threshold': optimal_threshold,
        }

    optimal_threshold = max(optimal_threshold, 0.05)
    y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    brier_score = float(np.mean((y_pred_proba - y_true) ** 2))

    return {
        'ROC-AUC': float(roc_auc_score(y_true, y_pred_proba)),
        'Optimal F1': float(f1_score(y_true, y_pred_binary, zero_division=0)),
        'Precision': float(precision_score(y_true, y_pred_binary, zero_division=0)),
        'Recall': float(recall_score(y_true, y_pred_binary, zero_division=0)),
        'Specificity': float(specificity),
        'Brier Score': brier_score,
        'Optimal Threshold': float(optimal_threshold),
        'Sample_Count': int(len(y_true)),
    }


def check_patient_leakage(df_train, df_val, df_test):
    train_ids = set(df_train['Patient ID'][df_train['Patient ID'].astype(str) != 'Unknown']) if df_train is not None else set()
    val_ids = set(df_val['Patient ID'][df_val['Patient ID'].astype(str) != 'Unknown']) if df_val is not None else set()
    test_ids = set(df_test['Patient ID'][df_test['Patient ID'].astype(str) != 'Unknown']) if df_test is not None else set()

    train_val = train_ids & val_ids
    train_test = train_ids & test_ids
    val_test = val_ids & test_ids

    print("\nPatient-level data leakage check (same patient appearing in more than one split):")
    print(f"  Train/Validation overlap: {len(train_val)} patient(s)")
    print(f"  Train/Testing overlap:    {len(train_test)} patient(s)")
    print(f"  Validation/Testing overlap: {len(val_test)} patient(s)")
    if train_val or train_test or val_test:
        print("  WARNING: overlapping patient IDs found across splits -- this is data leakage and will "
              "make validation/test performance look better than it really is.")
    else:
        print("  OK: no patient ID overlaps across Training/Validation/Testing.")

    return {
        'train_val_overlap': train_val,
        'train_test_overlap': train_test,
        'val_test_overlap': val_test,
    }


def load_data_from_folders(train_dir, val_dir, test_dir, comorbidity_groups, primary_target,
                            exclude_groups_from_labels=None):
    exclude_groups_from_labels = exclude_groups_from_labels or []
    class_names = list(comorbidity_groups)

    if primary_target not in class_names:
        print(f"Error: primary_target '{primary_target}' not found in comorbidity_groups.")
        return None, None, None, None, None, None, None, None, None
    pneumonia_class_index = class_names.index(primary_target)
    comorbidity_classes = [i for i in range(len(class_names)) if i != pneumonia_class_index]

    def _build_split(split_dir, split_name):
        if not split_dir or not os.path.isdir(split_dir):
            print(f"Warning: {split_name} directory not found at '{split_dir}'. Skipping.")
            return pd.DataFrame()

        rows = []
        n_total_seen = 0
        n_unparseable_age = 0
        sample_unparsed_filenames = []
        for group in class_names:
            group_dir = os.path.join(split_dir, group)
            if not os.path.isdir(group_dir):
                print(f"  Note: no '{group}' subfolder under {split_dir}.")
                continue
            for fname in os.listdir(group_dir):
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue
                n_total_seen += 1
                age = extract_patient_age(fname)
                if age is None:
                    n_unparseable_age += 1
                    if len(sample_unparsed_filenames) < 5:
                        sample_unparsed_filenames.append(fname)

                row = {
                    'Image Index': fname,
                    'Full Path': os.path.join(group_dir, fname),
                    'Group': group,
                    'Patient Age': age,
                    'Patient Gender': extract_patient_gender(fname),
                    'Image View': extract_image_view(fname),
                    'Patient ID': extract_patient_id(fname),
                }
                for g in class_names:
                    row[g] = 1 if g == group else 0
                rows.append(row)

        df = pd.DataFrame(rows)
        print(f"{split_name}: {n_total_seen} images found across {len(class_names)} group folders, "
              f"all {len(df)} loaded (no age-based filtering -- this cohort is already all-geriatric).")
        if n_total_seen > 0 and n_unparseable_age > 0:
            print(f"  Note: {n_unparseable_age}/{n_total_seen} images had an age that extract_patient_age() "
                  f"could not parse from the filename (recorded as 'Unknown' in the cohort description; "
                  f"these images are still included in training/evaluation). "
                  f"Examples: {sample_unparsed_filenames}")
        if exclude_groups_from_labels and len(df) > 0:
            before = len(df)
            df = df[~df['Group'].isin(exclude_groups_from_labels)].reset_index(drop=True)
            print(f"  Excluded {before - len(df)} images from group(s) {exclude_groups_from_labels}.")
        return df.reset_index(drop=True)

    print("\nBuilding datasets directly from Training/Validation/Testing folders (no CSV)...")
    df_train = _build_split(train_dir, "Training")
    df_val = _build_split(val_dir, "Validation")
    df_test = _build_split(test_dir, "Testing")
    print("------------------------------------------------------------------------")

    if len(df_train) == 0:
        print("Error: no training images found in the folder scan.")
        return None, None, None, None, None, None, None, None, None

    df_full = pd.concat([df_train, df_val, df_test], ignore_index=True) if len(df_train) else pd.DataFrame()
    multi_label = True

    pneumonia_count = df_train[primary_target].sum()
    non_pneumonia_count = len(df_train) - pneumonia_count
    pos_weight_value = (non_pneumonia_count / pneumonia_count) if pneumonia_count > 0 else MAX_POS_WEIGHT
    if pos_weight_value > MAX_POS_WEIGHT:
        print(f"Note: raw pos_weight ({pos_weight_value:.2f}) exceeds MAX_POS_WEIGHT ({MAX_POS_WEIGHT}); capping it "
              f"to avoid loss-scale instability from a small number of positive examples dominating the gradient.")
        pos_weight_value = MAX_POS_WEIGHT

    print(f"Pneumonia ('{primary_target}') Positives in Train Set: {pneumonia_count}")
    print(f"Pneumonia Negatives in Train Set: {non_pneumonia_count}")
    print(f"Pneumonia Loss Weight (pos_weight): {pos_weight_value:.2f}")

    for df in (df_full, df_train, df_val, df_test):
        if len(df) > 0 and 'Pneumonia' not in df.columns:
            df['Pneumonia'] = df[primary_target]

    return (df_full, df_train, df_val, df_test, multi_label, class_names,
            pneumonia_class_index, comorbidity_classes, pos_weight_value)


def compute_comorbidity_pos_weights(df_train, class_names, comorbidity_classes, max_weight=None):
    max_weight = max_weight if max_weight is not None else MAX_POS_WEIGHT
    weights = []
    print("\nComorbidity class balance (Train set) and computed pos_weight per class:")
    for idx in comorbidity_classes:
        group_name = class_names[idx]
        pos_count = df_train[group_name].sum()
        neg_count = len(df_train) - pos_count
        w = (neg_count / pos_count) if pos_count > 0 else max_weight
        w = min(w, max_weight)
        weights.append(w)
        print(f"  {group_name}: positives={pos_count}, negatives={neg_count}, pos_weight={w:.2f}")
    return torch.tensor(weights, dtype=torch.float32)


def load_data(data_csv_path):
    if not os.path.exists(data_csv_path):
        print(f"Error: CSV file not found at {data_csv_path}")
        return None, None, None, None, None, None

    df = pd.read_csv(data_csv_path)
    original_size = len(df)

    print("\nApplying geriatric filter (Patient Age >= 65)...")
    
    if 'Patient Age' not in df.columns:
        print("Error: 'Patient Age' column not found in CSV. Please verify column name.")
        return None, None, None, None, None, None
    
    df['Patient Age'] = pd.to_numeric(df['Patient Age'], errors='coerce').fillna(0) 
    df = df[df['Patient Age'] >= 65].copy()
    
    filtered_size = len(df)
    print(f"Original dataset size: {original_size}")
    print(f"Filtered geriatric size: {filtered_size}")
    
    if filtered_size == 0:
        print("Error: No data found for the geriatric age group (Age >= 65).")
        return None, None, None, None, None, None
            
    print("------------------------------------------------------------------------")
    
    df['Pathology'] = df['Finding Labels'].apply(lambda x: x.split('|'))
    
    mlb = MultiLabelBinarizer()
    labels_encoded = mlb.fit_transform(df['Pathology'])
    df_labels = pd.DataFrame(labels_encoded, columns=mlb.classes_)
    
    df = df.reset_index(drop=True) 
    df = pd.concat([df, df_labels], axis=1)

    multi_label = True
    class_names = mlb.classes_.tolist()
    
    try:
        pneumonia_class_index = class_names.index('Pneumonia')
    except ValueError:
        print("Warning: 'Pneumonia' class not found. Check your CSV data.")
        return None, None, None, None, None, None
        
    comorbidity_classes = [i for i, name in enumerate(class_names) if name != 'Pneumonia']

    df_train_full, df_temp = train_test_split(df, test_size=0.3, random_state=RANDOM_STATE, stratify=df['Pneumonia'])
    df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=df_temp['Pneumonia'])
    
    pneumonia_count = df_train_full['Pneumonia'].sum()
    non_pneumonia_count = len(df_train_full) - pneumonia_count
    pos_weight_value = non_pneumonia_count / pneumonia_count
    
    print(f"Pneumonia Positives in Train Set: {pneumonia_count}")
    print(f"Pneumonia Negatives in Train Set: {non_pneumonia_count}")
    print(f"Pneumonia Loss Weight (pos_weight): {pos_weight_value:.2f}")
    
    return df, df_train_full, df_val, df_test, multi_label, class_names, pneumonia_class_index, comorbidity_classes, pos_weight_value


def build_optimizer(model, lr=None, weight_decay=None, differential_lr=None, backbone_lr_mult=None):
    lr = lr if lr is not None else LEARNING_RATE
    weight_decay = weight_decay if weight_decay is not None else WEIGHT_DECAY
    differential_lr = differential_lr if differential_lr is not None else USE_DIFFERENTIAL_LR
    backbone_lr_mult = backbone_lr_mult if backbone_lr_mult is not None else BACKBONE_LR_MULTIPLIER

    if not differential_lr:
        return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    backbone_params = list(model.features.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith('features.')]

    return optim.Adam([
        {'params': backbone_params, 'lr': lr * backbone_lr_mult},
        {'params': head_params, 'lr': lr},
    ], weight_decay=weight_decay)


def train_model_enhanced(model, train_loader, val_loader, optimizer, num_epochs, multi_label, save_dir,
                          pneumonia_class_index, comorbidity_classes, pos_weight, ablation_mode=False,
                          viz_path=None, use_amp=None, patience=None, min_delta=None, use_lr_scheduler=None,
                          comorbidity_pos_weight=None, checkpoint_metric='val_loss'):
    """
    checkpoint_metric: 'val_loss' (default; minimize) or 'val_auc' (maximize). Both are
    tracked every epoch regardless of which one gates the saved checkpoint, so you can
    compare selection criteria after the fact without retraining.

    METRIC-DEFINITION NOTE:
    'train_acc' / 'val_acc_fixed_thresh_050' are computed at a FIXED 0.50 decision
    threshold on the raw sigmoid output. Because pneumonia_criterion uses a large
    pos_weight to correct for ~20% prevalence, raw probabilities are systematically
    pushed upward, especially in early epochs -- so this fixed-threshold accuracy can
    legitimately sit BELOW the ~80% "always predict negative" trivial baseline even
    while the model is learning real discriminative structure. It is kept only for
    backward-compatible plotting; do not use it to judge model quality or to describe
    validation performance in the manuscript.
    The metrics that actually match how the Results tables are computed (F1-optimal
    threshold, selected the same way evaluate_model() selects it at test time) are
    'val_auc', 'val_f1_optimal', and 'val_acc_optimal'. 
    """
    from sklearn.metrics import accuracy_score 

    if checkpoint_metric not in ('val_loss', 'val_auc'):
        raise ValueError("checkpoint_metric must be 'val_loss' or 'val_auc'")

    if use_amp is None:
        use_amp = torch.cuda.is_available()
    if patience is None:
        patience = EARLY_STOPPING_PATIENCE
    if min_delta is None:
        min_delta = EARLY_STOPPING_MIN_DELTA
    if use_lr_scheduler is None:
        use_lr_scheduler = LR_SCHEDULER_ENABLED

    n_train_samples = len(train_loader.dataset)
    n_val_samples = len(val_loader.dataset)
    n_train_batches = len(train_loader)
    n_val_batches = len(val_loader)
    print(f"Training on {n_train_samples} samples ({n_train_batches} batches/epoch), "
          f"validating on {n_val_samples} samples ({n_val_batches} batches/epoch), "
          f"batch_size={train_loader.batch_size}.")
    if n_train_batches < 5:
        print(f"WARNING: only {n_train_batches} training batches per epoch. This is very few weight "
              f"updates per epoch and is often why loss appears to plateau immediately -- check whether "
              f"n_train_samples ({n_train_samples}) is smaller than you expect for your actual dataset.")

    scheduler = None
    if use_lr_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_SCHEDULER_FACTOR, patience=LR_SCHEDULER_PATIENCE,
            min_lr=LR_SCHEDULER_MIN_LR
        )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc_fixed_thresh_050': [],
        'train_auc': [], 'val_auc': [],
        'val_f1_optimal': [], 'val_acc_optimal': [], 'val_opt_thresh': [],
        'checkpoint_metric': checkpoint_metric,
    }
    
    history['val_acc'] = history['val_acc_fixed_thresh_050']

    best_metric = float('inf') if checkpoint_metric == 'val_loss' else -float('inf')
    epochs_without_improvement = 0
    stopped_early = False

    pneumonia_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    if comorbidity_pos_weight is not None:
        comorbidity_criterion = nn.BCEWithLogitsLoss(pos_weight=comorbidity_pos_weight.to(device))
    else:
        comorbidity_criterion = nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_preds = 0
        total_preds = 0
        train_probs_epoch, train_labels_epoch = [], []

        if isinstance(train_loader.sampler, EnhancedPCISampler):
            train_loader.sampler.set_epoch(epoch)

        with tqdm(train_loader, unit='batch') as pbar:
            for i, (inputs, labels, pneumonia_labels, _) in enumerate(pbar):
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                pneumonia_labels = pneumonia_labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs, _, pneumonia_logits, comorbidity_logits = model(inputs)
                    pneumonia_loss = pneumonia_criterion(pneumonia_logits.squeeze(1), pneumonia_labels)
                    comorbidity_loss = comorbidity_criterion(comorbidity_logits, labels[:, comorbidity_classes])
                    loss = pneumonia_loss + comorbidity_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item()

                pneumonia_probs = torch.sigmoid(pneumonia_logits.squeeze(1)).detach()
                pneumonia_preds = pneumonia_probs > 0.5
                correct_preds += (pneumonia_preds.float() == pneumonia_labels).sum().item()
                total_preds += pneumonia_labels.size(0)
                train_probs_epoch.append(pneumonia_probs.cpu().numpy())
                train_labels_epoch.append(pneumonia_labels.cpu().numpy())

                pbar.set_description(f"Epoch {epoch+1}/{num_epochs}")
                pbar.set_postfix(loss=loss.item())

        epoch_train_loss = running_loss / len(train_loader)
        epoch_train_acc = correct_preds / total_preds
        train_probs_arr = np.concatenate(train_probs_epoch)
        train_labels_arr = np.concatenate(train_labels_epoch)
        epoch_train_auc = (roc_auc_score(train_labels_arr, train_probs_arr)
                            if len(np.unique(train_labels_arr)) > 1 else float('nan'))

        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['train_auc'].append(epoch_train_auc)

        model.eval()
        val_loss = 0.0
        val_correct_preds = 0
        val_total_preds = 0
        val_probs_epoch, val_labels_epoch = [], []

        with torch.no_grad():
            for inputs, labels, pneumonia_labels, _ in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                pneumonia_labels = pneumonia_labels.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs, _, pneumonia_logits, comorbidity_logits = model(inputs)
                    pneumonia_loss = pneumonia_criterion(pneumonia_logits.squeeze(1), pneumonia_labels)
                    comorbidity_loss = comorbidity_criterion(comorbidity_logits, labels[:, comorbidity_classes])
                    loss = pneumonia_loss + comorbidity_loss

                val_loss += loss.item()

                pneumonia_probs = torch.sigmoid(pneumonia_logits.squeeze(1)).detach()
                pneumonia_preds = pneumonia_probs > 0.5
                val_correct_preds += (pneumonia_preds.float() == pneumonia_labels).sum().item()
                val_total_preds += pneumonia_labels.size(0)
                val_probs_epoch.append(pneumonia_probs.cpu().numpy())
                val_labels_epoch.append(pneumonia_labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc_fixed = val_correct_preds / val_total_preds
        val_probs_arr = np.concatenate(val_probs_epoch)
        val_labels_arr = np.concatenate(val_labels_epoch)

        if len(np.unique(val_labels_arr)) > 1:
            epoch_val_auc = roc_auc_score(val_labels_arr, val_probs_arr)
            precision_arr, recall_arr, thresholds_arr = precision_recall_curve(val_labels_arr, val_probs_arr)
            f1_arr = (2 * precision_arr * recall_arr) / (precision_arr + recall_arr + 1e-10)
            ix = int(np.argmax(f1_arr))
            epoch_val_opt_thresh = float(thresholds_arr[ix]) if ix < len(thresholds_arr) else float(thresholds_arr[-1])
            epoch_val_opt_thresh = max(epoch_val_opt_thresh, 0.05)
            epoch_val_f1_optimal = float(f1_arr[ix])
            epoch_val_acc_optimal = float(accuracy_score(val_labels_arr, (val_probs_arr >= epoch_val_opt_thresh).astype(int)))
        else:
            epoch_val_auc = float('nan')
            epoch_val_opt_thresh = 0.5
            epoch_val_f1_optimal = 0.0
            epoch_val_acc_optimal = float('nan')

        history['val_loss'].append(epoch_val_loss)
        history['val_acc_fixed_thresh_050'].append(epoch_val_acc_fixed)
        history['val_auc'].append(epoch_val_auc)
        history['val_f1_optimal'].append(epoch_val_f1_optimal)
        history['val_acc_optimal'].append(epoch_val_acc_optimal)
        history['val_opt_thresh'].append(epoch_val_opt_thresh)

        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc@0.50: {epoch_train_acc:.4f}, Train AUC: {epoch_train_auc:.4f}, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc@0.50: {epoch_val_acc_fixed:.4f}, "
              f"Val AUC: {epoch_val_auc:.4f}, Val F1(opt): {epoch_val_f1_optimal:.4f}, Val Acc(opt thr): {epoch_val_acc_optimal:.4f}")

        if scheduler is not None:
            lr_before = optimizer.param_groups[0]['lr']
            scheduler.step(epoch_val_loss)
            lr_after = optimizer.param_groups[0]['lr']
            if lr_after < lr_before:
                print(f"Learning rate reduced: {lr_before:.2e} -> {lr_after:.2e}")

        current_metric = epoch_val_loss if checkpoint_metric == 'val_loss' else epoch_val_auc
        if checkpoint_metric == 'val_loss':
            improved = current_metric < best_metric - min_delta
        else:
            improved = (not np.isnan(current_metric)) and (current_metric > best_metric + min_delta)

        if improved:
            best_metric = current_metric
            history['best_epoch'] = epoch + 1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))
            print(f"Best model saved! (checkpoint_metric={checkpoint_metric}={best_metric:.4f})")
        else:
            epochs_without_improvement += 1
            if patience is not None and patience > 0:
                print(f"No improvement for {epochs_without_improvement}/{patience} epoch(s).")
                if epochs_without_improvement >= patience:
                    print(f"Early stopping triggered at epoch {epoch+1} (patience={patience}).")
                    stopped_early = True
                    break

    history['stopped_early'] = stopped_early
    history['epochs_trained'] = len(history['train_loss'])
    history['best_checkpoint_metric_value'] = best_metric
    history['patience_used'] = patience
    # v5: keep the full per-epoch history next to the checkpoint so stopping and selection can be audited later.
    try:
        with open(os.path.join(save_dir, 'history.json'), 'w') as fh:
            json.dump({k: (list(map(float, v)) if isinstance(v, list) else v) for k, v in history.items()
                       if k != 'val_acc'}, fh, indent=2, default=str)
    except Exception as e:
        print(f"Warning: could not save history.json ({e}).")

    if viz_path:
        try:
            os.makedirs(viz_path, exist_ok=True)
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))
            epochs_range = range(1, len(history['train_loss']) + 1)

            ax1.plot(epochs_range, history['train_loss'], lw=2, marker='o', label='Train Loss')
            ax1.plot(epochs_range, history['val_loss'], lw=2, marker='s', label='Val Loss')
            ax1.set_title('Training / Validation Loss', fontsize=18, fontweight='bold')
            ax1.set_xlabel('Epoch', fontsize=15, fontweight='bold')
            ax1.set_ylabel('Loss', fontsize=15, fontweight='bold')
            ax1.tick_params(axis='both', labelsize=13)
            ax1.legend(fontsize=13)
            ax1.grid(True, alpha=0.3)

            ax2.plot(epochs_range, history['train_acc'], lw=2, marker='o', label='Train Acc @0.50 (legacy)')
            ax2.plot(epochs_range, history['val_acc_fixed_thresh_050'], lw=2, marker='s', label='Val Acc @0.50 (legacy)')
            ax2.plot(epochs_range, history['val_acc_optimal'], lw=2, marker='^', label='Val Acc @F1-optimal thr')
            ax2.set_title('Accuracy (fixed vs. optimal threshold)', fontsize=18, fontweight='bold')
            ax2.set_xlabel('Epoch', fontsize=15, fontweight='bold')
            ax2.set_ylabel('Accuracy', fontsize=15, fontweight='bold')
            ax2.tick_params(axis='both', labelsize=13)
            ax2.legend(fontsize=11)
            ax2.grid(True, alpha=0.3)

            ax3.plot(epochs_range, history['train_auc'], lw=2, marker='o', label='Train AUC')
            ax3.plot(epochs_range, history['val_auc'], lw=2, marker='s', label='Val AUC')
            ax3.plot(epochs_range, history['val_f1_optimal'], lw=2, marker='^', label='Val F1 (optimal thr)')
            ax3.set_title('ROC-AUC / F1 (threshold-independent)', fontsize=18, fontweight='bold')
            ax3.set_xlabel('Epoch', fontsize=15, fontweight='bold')
            ax3.set_ylabel('Score', fontsize=15, fontweight='bold')
            ax3.tick_params(axis='both', labelsize=13)
            ax3.legend(fontsize=11)
            ax3.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(os.path.join(viz_path, 'training_curves.png'), dpi=300, bbox_inches='tight')
            plt.close()
            print(f"Training curves saved to {os.path.join(viz_path, 'training_curves.png')}")
        except Exception as e:
            print(f"Warning: could not save training curves ({e}).")

    best_ckpt = os.path.join(save_dir, 'best_model.pth')
    if os.path.exists(best_ckpt):
        model.load_state_dict(torch.load(best_ckpt, map_location=device))
    return model, history


def evaluate_model(model, loader, multi_label, pneumonia_class_index, save_path=None):
    model.eval()
    device = next(model.parameters()).device
    
    y_true_p = []
    y_scores_p = []
    total_loss = 0
    pneumonia_criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for inputs, labels, _, _ in tqdm(loader, desc="Evaluating"):
            inputs = inputs.to(device)
            labels = labels.to(device)

            all_outputs = model(inputs)
            pneumonia_logits = all_outputs[2].squeeze(1)

            pneumonia_labels = labels[:, pneumonia_class_index].float()

            total_loss += pneumonia_criterion(pneumonia_logits, pneumonia_labels).item()

            y_true_p.extend(pneumonia_labels.cpu().numpy())
            y_scores_p.extend(torch.sigmoid(pneumonia_logits).cpu().numpy())

    avg_val_loss = total_loss / len(loader)
    
    y_true_p = np.array(y_true_p)
    y_scores_p = np.array(y_scores_p)
    
    precision, recall, thresholds = precision_recall_curve(y_true_p, y_scores_p)
    
    fscore = (2 * precision * recall) / (precision + recall + 1e-10)
    ix = np.argmax(fscore)
    optimal_threshold = thresholds[ix] if ix < len(thresholds) else thresholds[-1]
    max_f1 = fscore[ix]
    
    print(f"Optimal Threshold (F1 Max): {optimal_threshold:.4f} | Max F1-Score: {max_f1:.4f}")
    
    if save_path:
        pass
    
    y_pred_p = (y_scores_p >= optimal_threshold).astype(int)
    
    avg_val_acc = np.mean(y_true_p == y_pred_p)
    
    return avg_val_loss, avg_val_acc, y_true_p, y_pred_p, y_scores_p, optimal_threshold

In [ ]:
class GradCAM:

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self._bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_index):
        self.model.zero_grad(set_to_none=True)
        logits, _, _, _ = self.model(input_tensor)
        score = logits[:, class_index].sum()
        score.backward()

        if self.activations is None or self.gradients is None:
            raise RuntimeError("GradCAM hooks did not fire; check that target_layer is part of the "
                                "forward graph reached from the model's output.")

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam).squeeze().cpu().numpy()

        cam_min, cam_max = cam.min(), cam.max()
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
        return cam

    def close(self):
        self._fwd_handle.remove()
        self._bwd_handle.remove()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()


def cam_to_bbox(cam, threshold_ratio=0.5):
    cam_resized = cv2.resize(cam, (224, 224))
    thresh = cam_resized.max() * threshold_ratio
    binary = (cam_resized >= thresh).astype(np.uint8)
    if binary.sum() == 0:
        return None
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    return (float(x), float(y), float(w), float(h))


def map_bbox_224_to_original(bbox, resize_size=None, crop_size=None, orig_size=1024):
    resize_size = resize_size if resize_size is not None else IMG_RESIZE_SIZE
    crop_size = crop_size if crop_size is not None else IMG_CROP_SIZE
    if bbox is None:
        return None
    x, y, w, h = bbox
    crop_offset = (resize_size - crop_size) / 2.0
    scale = orig_size / resize_size
    x0 = (x + crop_offset) * scale
    y0 = (y + crop_offset) * scale
    return (x0, y0, w * scale, h * scale)


def compute_iou(box_a, box_b):
    if box_a is None or box_b is None:
        return 0.0
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh

    inter_w = max(0.0, min(ax2, bx2) - max(ax, bx))
    inter_h = max(0.0, min(ay2, by2) - max(ay, by))
    inter_area = inter_w * inter_h
    union_area = aw * ah + bw * bh - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def load_bbox_annotations(bbox_csv_path, target_finding='Pneumonia'):
    if not bbox_csv_path or not os.path.exists(bbox_csv_path):
        print(f"Note: bounding-box annotation file not found at '{bbox_csv_path}'. "
              f"IoU/localization metrics will be skipped.")
        return None
    raw = pd.read_csv(bbox_csv_path)
    cols = list(raw.columns)
    if len(cols) < 6:
        print("Warning: bounding-box CSV has an unexpected number of columns; skipping IoU evaluation.")
        return None
    rename_map = {cols[0]: 'Image Index', cols[1]: 'Finding Label',
                  cols[2]: 'x', cols[3]: 'y', cols[4]: 'w', cols[5]: 'h'}
    raw = raw.rename(columns=rename_map)
    raw = raw[['Image Index', 'Finding Label', 'x', 'y', 'w', 'h']].dropna(subset=['Image Index'])
    raw['Finding Label'] = raw['Finding Label'].astype(str).str.strip()
    filtered = raw[raw['Finding Label'].str.lower() == target_finding.lower()].reset_index(drop=True)
    print(f"Loaded {len(filtered)} ground-truth '{target_finding}' bounding boxes "
          f"(from {len(raw)} total annotated boxes).")
    return filtered if len(filtered) > 0 else None




In [ ]:
def create_central_region_mask(image_shape, center_ratio=0.5):
    if len(image_shape) == 3:
        h, w = image_shape[:2]
    else:
        h, w = image_shape
    center_h, center_w = h // 2, w // 2
    crop_h, crop_w = int(h * center_ratio), int(w * center_ratio)

    mask = np.zeros((h, w), dtype=np.uint8)
    y_start = max(0, center_h - crop_h // 2)
    y_end = min(h, center_h + crop_h // 2)
    x_start = max(0, center_w - crop_w // 2)
    x_end = min(w, center_w + crop_w // 2)
    mask[y_start:y_end, x_start:x_end] = 1
    return mask


def compute_mask_iou(heatmap, mask, threshold=0.5):
    binary_heatmap = (heatmap >= threshold).astype(np.uint8)
    binary_mask = (mask > 0).astype(np.uint8)

    intersection = np.logical_and(binary_heatmap, binary_mask).sum()
    union = np.logical_or(binary_heatmap, binary_mask).sum()
    if union == 0:
        return 0.0, 0.0, 0.0

    iou = intersection / union
    tp = intersection
    fp = binary_heatmap.sum() - tp
    fn = binary_mask.sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return float(iou), float(precision), float(recall)


def compute_optimal_iou_threshold(heatmap, mask, thresholds=np.arange(0.1, 0.95, 0.05)):
    best_iou, best_threshold = 0.0, 0.5
    for t in thresholds:
        iou, _, _ = compute_mask_iou(heatmap, mask, t)
        if iou > best_iou:
            best_iou, best_threshold = iou, t
    return float(best_threshold), float(best_iou)


def compute_attention_entropy(heatmap):
    heatmap_norm = heatmap / (heatmap.sum() + 1e-8)
    flat = heatmap_norm.flatten()
    flat = flat[flat > 0]
    if len(flat) == 0:
        return 0.0
    return float(-np.sum(flat * np.log2(flat + 1e-8)))


def compute_peak_to_mean_ratio(heatmap):
    mean_attention = np.mean(heatmap)
    return float(np.max(heatmap) / mean_attention) if mean_attention > 0 else 0.0


def compute_attention_coverage(heatmap, threshold=0.5):
    return float(np.sum(heatmap > threshold) / heatmap.size)


def compute_number_of_attention_peaks(heatmap, threshold_percentile=90):
    from scipy import ndimage
    thresh_value = np.percentile(heatmap, threshold_percentile)
    binary = (heatmap >= thresh_value).astype(np.uint8)
    _, num_features = ndimage.label(binary)
    return int(num_features)


def compute_attention_dispersion(heatmap):
    h, w = heatmap.shape
    y, x = np.mgrid[0:h, 0:w]
    heatmap_norm = heatmap / (heatmap.sum() + 1e-8)
    cx = np.sum(x * heatmap_norm)
    cy = np.sum(y * heatmap_norm)
    return float(np.sum(((x - cx) ** 2 + (y - cy) ** 2) * heatmap_norm))


def compute_attention_quality_metrics(heatmap):
    heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return {
        'entropy': compute_attention_entropy(heatmap_norm),
        'peak_to_mean_ratio': compute_peak_to_mean_ratio(heatmap_norm),
        'coverage_05': compute_attention_coverage(heatmap_norm, threshold=0.5),
        'n_peaks': compute_number_of_attention_peaks(heatmap_norm, threshold_percentile=90),
        'dispersion': compute_attention_dispersion(heatmap_norm),
    }




## P1. Configuration

In [ ]:
# ============================================================================
# P1. PROTOCOL-2 CONFIGURATION (frozen by hash; every run records it)
# ============================================================================
import hashlib, json, time, math, random, re, copy as _copy

P2_STAGE = 'confirm'        # 'dev' while you are still tuning on VALIDATION; 'confirm' for the pre-registered runs

P2_CFG = {
    'protocol': 'P2-spatialPAAG-boundedCSM',
    # ---- data / preprocessing
    'img_size': 320,                       # network input (DenseNet-121 -> 10x10 feature grid)
    'cache_short_side': None,              # None -> round(img_size * 256 / 224), same ratio as Protocol 1
    'clahe_clip': 2.0, 'clahe_grid': 8,
    'aug_scale': (0.75, 1.0), 'aug_ratio': (0.9, 1.1),
    'aug_degrees': 7.0, 'aug_translate': 0.04, 'aug_brightness': 0.10, 'aug_contrast': 0.10,
    'aug_hflip': False,                    # cardiac laterality: no horizontal flips
    # ---- labels
    'pneumonia_label': 'consensus',        # cohort v2: 'consensus' (both CheXpert labelers agree; disagreements masked),
                                           # 'image' (VisualCheXbert only = Protocol-1 definition), 'report' (report-positive
                                           # vs consensus negatives). Ignored for the old folder cohort.
    'aux_obs': True, 'lambda_aux': 0.5,    # auxiliary heads for the 13 other CheXpert findings (cohort v2 only)
    'comorbidity_labels': 'partial',       # old folder cohort only: 'partial' (priority-aware) or 'single'
    'use_multilabel_csv': True,            # fill unknowns from NIH / CheXpert label files when they are available
    # ---- model
    'backbone_init': 'imagenet',           # 'imagenet' or chest-X-ray weights that NEVER saw CheXpert/NIH:
                                           # 'xrv:densenet121-res224-mimic_ch' (MIMIC-CXR) / 'xrv:densenet121-res224-pc' (PadChest)
    'head_dropout': 0.0,
    'paag_hidden': 256, 'paag_init_bias': -3.0,
    'csm_hidden': 32, 'csm_init_bias': -3.0, 'csm_detach': True,
    # ---- optimisation
    'epochs': 6, 'warmup_epochs': 1, 'patience': 3,   # frozen after the development sweep (P6b)
    'batch_size': 48, 'lr': 3e-4, 'backbone_lr_mult': 0.1, 'weight_decay': 1e-4,
    'grad_clip': 1.0, 'ema_decay': 0.999, 'min_lr_frac': 0.01,
    'imbalance': 'pos_weight',             # single correction
    'pos_weight_mode': 'full',             # 'full' (neg/pos) or 'sqrt' (gentler; less memorisation of rare positives)
    'max_pos_weight': 20.0, 'lambda_com': 1.0,
    'checkpoint_metric': 'val_auc_ema',
    # ---- seeds / analysis plan
    'seed_pool_size': 40, 'n_target_seeds': 20,
    'primary_contrasts': [['paag', 'base'], ['csm', 'base'], ['paag_csm', 'base']],
    'fpr_spec_target': 0.80,
}
if P2_CFG['cache_short_side'] is None:
    P2_CFG['cache_short_side'] = int(round(P2_CFG['img_size'] * 256 / 224))

# Paths and machine settings do not affect the science, so they are NOT part of the hash.
P2_PATHS = {
    'results_root': os.path.join(BASE_DIR, 'results_P2'),
    # Cohort v2 (collision-free, multi-label) from cohort_builder_v2.py. If this file exists it REPLACES the old
    # Training/Validation/Testing folders.
    'cohort_manifest': os.path.join(BASE_DIR, 'cohort_v2', 'cohort_v2_manifest.csv'),
    'cohort_images': os.path.join(BASE_DIR, 'cohort_v2', 'images'),
    'cache_dir': os.path.join(BASE_DIR, f"cache_clahe_{P2_CFG['cache_short_side']}"),
    # Optional label sources for true multi-label comorbidity targets (leave None if not available):
    'nih_data_entry_csv': None,            # e.g. '/notebooks/meta/Data_Entry_2017_v2020.csv'
    'chexpert_labels_csv': None,           # VisualCheXbert labels with a 'Path' column
    'chexpert_meta_csv': None,             # CheXpert train.csv (Path, Sex, Age, AP/PA) to disambiguate studies
    'chexpert_path_map_csv': None,         # optional explicit map: 'Image Index' -> 'Path' (from your cohort notebook)
    'torchxrayvision_cache': os.path.join(BASE_DIR, '.torchxrayvision'),
}
P2_MACHINE = {'num_workers': min(8, os.cpu_count() or 4), 'amp': torch.cuda.is_available(),
              'channels_last': torch.cuda.is_available(), 'session_hours': 5.5}

P2_VARIANTS = {
    'base':     {'paag': False, 'csm': False, 'label': 'Base'},
    'paag':     {'paag': True,  'csm': False, 'label': 'PAAG'},
    'csm':      {'paag': False, 'csm': True,  'label': 'CSM'},
    'paag_csm': {'paag': True,  'csm': True,  'label': 'Full model'},
}


_P2_LEGACY_DEFAULTS = {'backbone_init': 'imagenet', 'head_dropout': 0.0, 'pos_weight_mode': 'full'}


def p2_config_hash(cfg=None):
    cfg = P2_CFG if cfg is None else cfg
    # options added in v9 are left out of the hash while at their old behaviour, so earlier runs keep their hash
    cfg = {k: v for k, v in cfg.items() if not (k in _P2_LEGACY_DEFAULTS and v == _P2_LEGACY_DEFAULTS[k])}
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:12]


P2_HASH = p2_config_hash()
os.makedirs(P2_PATHS['results_root'], exist_ok=True)
print(f"Protocol 2 | stage = {P2_STAGE} | config hash = {P2_HASH}")
print(f"  img {P2_CFG['img_size']} px (cache short side {P2_CFG['cache_short_side']}), epochs {P2_CFG['epochs']}, "
      f"batch {P2_CFG['batch_size']}, lr {P2_CFG['lr']}, EMA {P2_CFG['ema_decay']}, checkpoint = {P2_CFG['checkpoint_metric']}")
print(f"  results -> {P2_PATHS['results_root']}")


## P1b. Label the existing folders from the original CSVs

In [ ]:
# ============================================================================
# P1b. LABEL THE EXISTING FOLDERS FROM THE ORIGINAL CSVs (no re-export, no re-upload)
# ----------------------------------------------------------------------------
# Your CheXpert filenames have no study ID, so one file on disk can stand for several studies of the same
# patient (same age, sex, view and view number). The candidates are known: every original label row with that
# key AND the same diagnostic group as the file's folder. For each label we use the candidates' value when they
# all agree (exact), and -1 (unknown -> masked in the loss and in evaluation) when they disagree. NIH filenames
# contain the image index, so NIH labels are always exact.
# Output: cohort_from_folders.csv in the cohort-v2 format, read by P2 exactly like a rebuilt cohort.
# Put the three CSVs in /notebooks/CSVs (or edit P2_CSV below). Runs once (~1 min); later sessions reuse it.
# ============================================================================
import re
P2_CSV = {'visual': os.path.join(BASE_DIR, 'CSVs', 'train_visualCheXbert.csv'),
          'report': os.path.join(BASE_DIR, 'CSVs', 'train_cheXbert.csv'),
          'nih': os.path.join(BASE_DIR, 'CSVs', 'Data_Entry_2017_v2020.csv')}
P2_FOLDER_COHORT = os.path.join(BASE_DIR, 'CSVs', 'cohort_from_folders.csv')

_CXP_OBS = ['Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
            'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
            'No Finding']
_COH_GROUPS = {'Pneumonia_Infectious': 'Pneumonia (Infectious)', 'Cardiomegaly': 'Cardiomegaly',
               'Pleural_Extrapulmonary_Volume_Disease': 'Pleural/Extrapulmonary/Volume Disease',
               'Weakly_Supervised_Uncertain': 'Airspace Opacity (non-infectious)',
               'Focal_Lesion_Chronic_Change': 'Focal Lesion/Chronic Change', 'No_Finding': 'No Finding'}
_COH_COMORB = ['Cardiomegaly', 'Pleural/Extrapulmonary/Volume Disease', 'Airspace Opacity (non-infectious)',
               'Focal Lesion/Chronic Change', 'No Finding']
_NIH_GRP = {'Cardiomegaly': 'Cardiomegaly', 'Atelectasis': 'Pleural/Extrapulmonary/Volume Disease',
            'Effusion': 'Pleural/Extrapulmonary/Volume Disease', 'Hernia': 'Pleural/Extrapulmonary/Volume Disease',
            'Pleural_Thickening': 'Pleural/Extrapulmonary/Volume Disease',
            'Pneumothorax': 'Pleural/Extrapulmonary/Volume Disease', 'Consolidation': 'Airspace Opacity (non-infectious)',
            'Edema': 'Airspace Opacity (non-infectious)', 'Infiltration': 'Airspace Opacity (non-infectious)',
            'Emphysema': 'Focal Lesion/Chronic Change', 'Fibrosis': 'Focal Lesion/Chronic Change',
            'Mass': 'Focal Lesion/Chronic Change', 'Nodule': 'Focal Lesion/Chronic Change'}
_NIH_OBS = {'Cardiomegaly': 'Cardiomegaly', 'Edema': 'Edema', 'Consolidation': 'Consolidation',
            'Atelectasis': 'Atelectasis', 'Pneumothorax': 'Pneumothorax', 'Effusion': 'Pleural Effusion',
            'Mass': 'Lung Lesion', 'Nodule': 'Lung Lesion', 'No Finding': 'No Finding'}


def _cxp_rule_group(v):
    """Vectorised version of the cohort's priority rule on VisualCheXbert labels (folder-name spelling)."""
    g = np.full(len(v), None, dtype=object)
    pleural = v[['Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other']].eq(1).any(axis=1).values
    for cond, name in [(v['No Finding'].eq(1).values, 'No_Finding'), (v['Edema'].eq(1).values, 'Weakly_Supervised_Uncertain'),
                       (pleural, 'Pleural_Extrapulmonary_Volume_Disease'), (v['Cardiomegaly'].eq(1).values, 'Cardiomegaly'),
                       (v['Pneumonia'].eq(1).values, 'Pneumonia_Infectious')]:      # reverse priority: last wins
        g[cond] = name
    return g


def p2_label_existing_folders(force=False):
    if os.path.exists(P2_FOLDER_COHORT) and not force:
        print(f"Using existing {P2_FOLDER_COHORT}")
        return pd.read_csv(P2_FOLDER_COHORT, low_memory=False)
    missing = [p for p in P2_CSV.values() if not os.path.exists(p)]
    if missing:
        print(f"CSV(s) not found: {missing} -> P2 will use the folder labels only."); return None
    # ---- files on disk
    rows = []
    for split_dir, split in [(TRAIN_DIR, 'Train'), (VAL_DIR, 'Validation'), (TEST_DIR, 'Test')]:
        for g in sorted(os.listdir(split_dir)):
            gd = os.path.join(split_dir, g)
            if os.path.isdir(gd) and g in _COH_GROUPS:
                for f in sorted(os.listdir(gd)):
                    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                        rows.append((split, g, f, os.path.relpath(os.path.join(gd, f), BASE_DIR)))
    files = pd.DataFrame(rows, columns=['split', 'folder_group', 'fname', 'file_rel'])
    # ---- CheXpert candidates
    vis = pd.read_csv(P2_CSV['visual'])
    rep = pd.read_csv(P2_CSV['report'])[['Path', 'Pneumonia']].rename(columns={'Pneumonia': 'rep_pneu'})
    v = vis[(vis['Frontal/Lateral'] == 'Frontal') & (vis['Age'] >= 65)].merge(rep, on='Path', how='left')
    v['pid'] = v['Path'].str.extract(r'(patient\d+)', expand=False)
    v['viewfile'] = v['Path'].str.extract(r'(view\d+_frontal)', expand=False)
    v['folder_group'] = _cxp_rule_group(v)
    v = v.dropna(subset=['folder_group'])
    v['fname'] = ('CheXpert_' + v['pid'] + '_' + v['Age'].astype(int).astype(str) + '_' + v['Sex'] + '_' +
                  v['AP/PA'].astype(str) + '_' + v['viewfile'] + '.jpg')
    v['y_image'] = v['Pneumonia'].eq(1).astype(int)
    rp = v['rep_pneu']
    v['y_report'] = np.select([rp.eq(1), rp.eq(-1), rp.eq(0)], [1, -1, 0], 9)
    rpos = rp.isin([1.0, -1.0])
    v['y_consensus'] = np.where(v['y_image'].eq(1) & rpos, 1, np.where(v['y_image'].eq(0) & ~rpos, 0, -1))
    v['com::Cardiomegaly'] = v['Cardiomegaly'].eq(1).astype(int)
    v['com::Pleural/Extrapulmonary/Volume Disease'] = v[['Atelectasis', 'Pneumothorax', 'Pleural Effusion',
                                                          'Pleural Other']].eq(1).any(axis=1).astype(int)
    v['com::Airspace Opacity (non-infectious)'] = v['Edema'].eq(1).astype(int)
    v['com::No Finding'] = v['No Finding'].eq(1).astype(int)
    for o in _CXP_OBS:
        v['obs::' + o] = v[o].eq(1).astype(int)
    lab_cols = ['y_image', 'y_report', 'y_consensus'] + [c for c in v.columns if c.startswith(('com::', 'obs::'))]
    g = v.groupby(['fname', 'folder_group'])
    first, nuniq = g[lab_cols].first(), g[lab_cols].nunique()
    agg = first.where(nuniq.eq(1), -1)                     # candidates disagree -> unknown
    agg['y_report'] = first['y_report'].where(nuniq['y_report'].eq(1), -2)   # -2: report labels disagree
    agg['n_candidates'] = g.size()
    agg['age'] = g['Age'].first()
    agg = agg.reset_index()
    agg['com::Focal Lesion/Chronic Change'] = -1
    cx = files[files['fname'].str.startswith('CheXpert')].merge(agg, on=['fname', 'folder_group'], how='left')
    # ---- NIH (exact)
    de = pd.read_csv(P2_CSV['nih'])
    lab = dict(zip(de['Image Index'], de['Finding Labels'].astype(str)))
    nx = files[files['fname'].str.startswith('ChestX-ray8')].copy()
    nrows = []
    for f in nx['fname']:
        m = re.search(r'_(\d{8}_\d{3})\.png$', f)
        fs = set(lab.get(m.group(1) + '.png', 'NA').split('|')) if m else {'NA'}
        r = {'y_image': int('Pneumonia' in fs), 'y_report': int('Pneumonia' in fs), 'y_consensus': int('Pneumonia' in fs),
             'n_candidates': 1 if m and m.group(1) + '.png' in lab else 0}
        for g in _COH_COMORB:
            r['com::' + g] = int(fs == {'No Finding'}) if g == 'No Finding' else int(any(_NIH_GRP.get(z) == g for z in fs))
        for o in _CXP_OBS:
            r['obs::' + o] = int(any(_NIH_OBS.get(z) == o for z in fs)) if o in set(_NIH_OBS.values()) else -1
        nrows.append(r)
    nx = pd.concat([nx.reset_index(drop=True), pd.DataFrame(nrows)], axis=1)
    coh = pd.concat([cx, nx], ignore_index=True)
    meta = coh['fname'].apply(parse_filename_metadata)
    coh['source'] = [m['source'] for m in meta]
    coh['patient_id'] = [m['patient_id'] for m in meta]
    coh['view'] = [m['view'] for m in meta]
    coh['sex'] = [m['sex'] for m in meta]
    coh['age'] = coh['age'].fillna(pd.Series([m['age'] for m in meta]))
    coh['group'] = coh['folder_group'].map(_COH_GROUPS)
    coh['image_id'] = coh['folder_group'] + '__' + coh['fname'].str.rsplit('.', n=1).str[0]
    coh['ext'] = '.' + coh['fname'].str.rsplit('.', n=1).str[1]
    unmatched = coh['n_candidates'].fillna(0).eq(0)
    for c in [c for c in coh.columns if c.startswith(('y_', 'com::', 'obs::'))]:
        coh[c] = coh[c].fillna(-1).astype(int)
    # Folder labels stay authoritative for y_image (it defined the folder); keep it even if unmatched
    coh['y_image'] = (coh['folder_group'] == 'Pneumonia_Infectious').astype(int)
    assert coh['image_id'].is_unique
    coh.to_csv(P2_FOLDER_COHORT, index=False)
    cxm = coh['source'] == 'CheXpert'
    print(f"Labelled {len(coh)} existing files ({int(cxm.sum())} CheXpert, {int((~cxm).sum())} ChestX-ray8); "
          f"unmatched to any CSV row: {int(unmatched.sum())}")
    print(f"  CheXpert files standing for exactly one study: {int((coh.loc[cxm, 'n_candidates'] == 1).sum())} | "
          f"for several studies: {int((coh.loc[cxm, 'n_candidates'] > 1).sum())}")
    print("  consensus pneumonia label 1 / 0 / masked by split:\n" +
          pd.crosstab(coh['y_consensus'], coh['split']).to_string())
    known = coh.loc[cxm & coh['y_consensus'].eq(1), 'com::Cardiomegaly']
    print(f"  consensus-pneumonia CheXpert files with cardiomegaly known & present: {int((known == 1).sum())} of {len(known)}")
    return coh


_p2_folder_coh = p2_label_existing_folders()
if _p2_folder_coh is not None and not os.path.exists(P2_PATHS['cohort_manifest']):
    P2_PATHS['cohort_manifest'] = P2_FOLDER_COHORT       # P2 now reads labels from the CSVs ...
    P2_PATHS['cohort_images'] = BASE_DIR                 # ... and images from the existing folders
    print("P2 will train on the EXISTING folders with CSV-derived labels.")


## P2. Data, cache and comorbidity targets

In [ ]:
# ============================================================================
# P2. DATA: split tables, one-off CLAHE cache, comorbidity targets (partial / multi-label)
# ============================================================================
from concurrent.futures import ThreadPoolExecutor

P2_GROUP_PRIORITY = ['Pneumonia_Infectious', 'Cardiomegaly', 'Pleural_Extrapulmonary_Volume_Disease',
                     'Weakly_Supervised_Uncertain', 'Focal_Lesion_Chronic_Change', 'No_Finding']
P2_COMORB = ['Cardiomegaly', 'Focal_Lesion_Chronic_Change', 'No_Finding',
             'Pleural_Extrapulmonary_Volume_Disease', 'Weakly_Supervised_Uncertain']   # order of the 5 heads

# ---------------------------------------------------------------- split tables
_P2_COHORT = bool(P2_PATHS.get('cohort_manifest')) and os.path.exists(P2_PATHS['cohort_manifest'])
_G_MAP = {'Pneumonia (Infectious)': 'Pneumonia_Infectious', 'Cardiomegaly': 'Cardiomegaly',
          'Pleural/Extrapulmonary/Volume Disease': 'Pleural_Extrapulmonary_Volume_Disease',
          'Airspace Opacity (non-infectious)': 'Weakly_Supervised_Uncertain',
          'Focal Lesion/Chronic Change': 'Focal_Lesion_Chronic_Change', 'No Finding': 'No_Finding'}
_COM_MAP = {v: k for k, v in _G_MAP.items() if v != 'Pneumonia_Infectious'}
_p2_split = {}
P2_AUX = []
if _P2_COHORT:
    _coh = pd.read_csv(P2_PATHS['cohort_manifest'], low_memory=False)
    P2_AUX = [c[5:] for c in _coh.columns if c.startswith('obs::')] if P2_CFG['aux_obs'] else []
    for _k, _s in [('train', 'Train'), ('val', 'Validation'), ('test', 'Test')]:
        d = _coh[_coh['split'] == _s].reset_index(drop=True)
        if P2_CFG['pneumonia_label'] == 'consensus':
            y = d['y_consensus'].values
        elif P2_CFG['pneumonia_label'] == 'image':
            y = d['y_image'].values
        elif P2_CFG['pneumonia_label'] == 'report':
            y = np.where(d['y_report'] == 1, 1, np.where(d['y_consensus'] == 0, 0, -1))
        else:
            raise ValueError(P2_CFG['pneumonia_label'])
        # existing-folder cohort keeps the on-disk filename, so the CLAHE cache already built is reused as is
        _idx = d['fname'] if 'fname' in d.columns else d['image_id'] + d['ext']
        df = pd.DataFrame({'Image Index': _idx.values,
                           'Full Path': [os.path.join(P2_PATHS['cohort_images'], f) for f in d['file_rel']],
                           'Group': d['group'].map(_G_MAP).values, 'Patient ID': d['patient_id'].astype(str).values,
                           'Source': d['source'].values, 'Split': _k, 'Image View': d['view'].values,
                           'Patient Age': d['age'].values, 'Pneumonia': y.astype(int),
                           'y_image': d['y_image'].values, 'y_consensus': d['y_consensus'].values,
                           'y_report': d['y_report'].values})
        for c in P2_COMORB:
            df[f'com_{c}'] = d['com::' + _COM_MAP[c]].values.astype(np.int8)
        for o in P2_AUX:
            df[f'aux_{o}'] = d['obs::' + o].values.astype(np.int8)
        _p2_split[_k] = df
    _missing = [p for p in _p2_split['train']['Full Path'].head(200) if not os.path.exists(p)]
    print(f"Cohort v2 loaded ({P2_CFG['pneumonia_label']} pneumonia label, {len(P2_AUX)} auxiliary findings). "
          f"Missing image files among the first 200: {len(_missing)}")
    for _k, _df in _p2_split.items():
        print(f"  {_k}: {len(_df)} images | pneumonia 1/0/masked = {int((_df.Pneumonia == 1).sum())}/"
              f"{int((_df.Pneumonia == 0).sum())}/{int((_df.Pneumonia < 0).sum())}")
else:
    (_, _p2_split['train'], _p2_split['val'], _p2_split['test'], *_rest) = load_data_from_folders(
        TRAIN_DIR, VAL_DIR, TEST_DIR, COMORBIDITY_GROUPS, PRIMARY_TARGET)
    for _k, _df in _p2_split.items():
        _df = _df.reset_index(drop=True)
        _meta = _df['Image Index'].apply(parse_filename_metadata)
        _df['Source'] = [m['source'] for m in _meta]
        _df['Split'] = _k
        _p2_split[_k] = _df
    print("Old folder cohort (no cohort_v2_manifest.csv found):", {k: len(v) for k, v in _p2_split.items()})


# ---------------------------------------------------------------- CLAHE cache (one-off; ~minutes with threads)
def _p2_cache_path(row):
    return os.path.join(P2_PATHS['cache_dir'], row['Split'], row['Group'],
                        os.path.splitext(row['Image Index'])[0] + '.png')


_p2_clahe = cv2.createCLAHE(clipLimit=P2_CFG['clahe_clip'], tileGridSize=(P2_CFG['clahe_grid'], P2_CFG['clahe_grid']))


def _p2_cache_one(args):
    src, dst = args
    if os.path.exists(dst):
        return 0
    # PIL, not cv2: identical to Protocol 1's CLAHETransform (PIL .convert('L')), and it silently ignores the
    # iCCP colour-profile chunk in some NIH PNGs that makes libpng print a warning for every file.
    with Image.open(src) as im:
        img = np.array(im.convert('L'))
    img = cv2.createCLAHE(clipLimit=P2_CFG['clahe_clip'],
                          tileGridSize=(P2_CFG['clahe_grid'], P2_CFG['clahe_grid'])).apply(img)  # same order as P1
    h, w = img.shape
    s = P2_CFG['cache_short_side'] / min(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    cv2.imwrite(dst, img, [cv2.IMWRITE_PNG_COMPRESSION, 1])
    return 1


def p2_build_cache():
    jobs = [(r['Full Path'], _p2_cache_path(r)) for df in _p2_split.values() for _, r in df.iterrows()]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max(2, P2_MACHINE['num_workers'])) as ex:
        n_new = sum(ex.map(_p2_cache_one, jobs, chunksize=64))
    print(f"CLAHE cache: {len(jobs)} images ({n_new} newly written) in {time.time() - t0:.0f}s -> {P2_PATHS['cache_dir']}")


p2_build_cache()
for _df in _p2_split.values():
    _df['Cache Path'] = _df.apply(_p2_cache_path, axis=1)


# ---------------------------------------------------------------- comorbidity targets
def p2_partial_targets(group):
    """Priority-aware partial labels. The single-label folders were built by a priority rule, so for an image in
    group g: every HIGHER-priority group is known absent (0), g itself is present (1), and every LOWER-priority
    group is unknown (-1, masked in the loss). No_Finding is 1 only for No_Finding images and 0 otherwise."""
    rank = {g: i for i, g in enumerate(P2_GROUP_PRIORITY)}
    out = {}
    for c in P2_COMORB:
        if c == 'No_Finding':
            out[c] = 1 if group == 'No_Finding' else 0
        elif c == group:
            out[c] = 1
        elif rank[c] < rank[group]:
            out[c] = 0
        else:
            out[c] = -1
    return out


_NIH_TO_GROUP = {'Cardiomegaly': 'Cardiomegaly', 'Atelectasis': 'Pleural_Extrapulmonary_Volume_Disease',
                 'Effusion': 'Pleural_Extrapulmonary_Volume_Disease', 'Hernia': 'Pleural_Extrapulmonary_Volume_Disease',
                 'Pleural_Thickening': 'Pleural_Extrapulmonary_Volume_Disease',
                 'Pneumothorax': 'Pleural_Extrapulmonary_Volume_Disease', 'Consolidation': 'Weakly_Supervised_Uncertain',
                 'Edema': 'Weakly_Supervised_Uncertain', 'Infiltration': 'Weakly_Supervised_Uncertain',
                 'Emphysema': 'Focal_Lesion_Chronic_Change', 'Fibrosis': 'Focal_Lesion_Chronic_Change',
                 'Mass': 'Focal_Lesion_Chronic_Change', 'Nodule': 'Focal_Lesion_Chronic_Change'}
_CXP_TO_GROUP = {'Cardiomegaly': 'Cardiomegaly', 'Atelectasis': 'Pleural_Extrapulmonary_Volume_Disease',
                 'Pneumothorax': 'Pleural_Extrapulmonary_Volume_Disease',
                 'Pleural Effusion': 'Pleural_Extrapulmonary_Volume_Disease',
                 'Pleural Other': 'Pleural_Extrapulmonary_Volume_Disease', 'Edema': 'Weakly_Supervised_Uncertain'}
# CheXpert has no label that maps to the focal-lesion group -> that target stays unknown for CheXpert images.


def _nih_index(fname):
    m = re.search(r'_(\d{8})_(\d{3})\.png$', fname)
    return f"{m.group(1)}_{m.group(2)}.png" if m else None


def _p2_multilabel_lookup():
    """Image Index -> {group: 0/1/-1} from the original label files, when available."""
    out = {}
    p = P2_PATHS.get('nih_data_entry_csv')
    if P2_CFG['use_multilabel_csv'] and p and os.path.exists(p):
        de = pd.read_csv(p)
        lab = dict(zip(de['Image Index'], de['Finding Labels'].astype(str)))
        n = 0
        for df in _p2_split.values():
            for fname in df.loc[df['Source'] == 'ChestX-ray8', 'Image Index']:
                key = _nih_index(fname)
                if key in lab:
                    finds = set(lab[key].split('|'))
                    tgt = {c: 0 for c in P2_COMORB}
                    for f in finds:
                        if f in _NIH_TO_GROUP:
                            tgt[_NIH_TO_GROUP[f]] = 1
                    tgt['No_Finding'] = 1 if finds == {'No Finding'} else 0
                    out[fname] = tgt; n += 1
        print(f"NIH multi-labels matched for {n} images")
    p, pm = P2_PATHS.get('chexpert_labels_csv'), P2_PATHS.get('chexpert_path_map_csv')
    if P2_CFG['use_multilabel_csv'] and p and os.path.exists(p):
        cl = pd.read_csv(p)
        cl['Path'] = cl['Path'].astype(str)
        path_for = {}
        if pm and os.path.exists(pm):
            mp = pd.read_csv(pm)
            path_for = dict(zip(mp['Image Index'], mp['Path'].astype(str)))
        else:
            # Disambiguate by patient + view file, then (if CheXpert train.csv is given) by age and AP/PA.
            cl['pid'] = cl['Path'].str.extract(r'(patient\d+)', expand=False).str.lower()
            cl['viewfile'] = cl['Path'].str.extract(r'(view\d+_frontal)', expand=False)
            meta = P2_PATHS.get('chexpert_meta_csv')
            have_meta = bool(meta and os.path.exists(meta))
            if have_meta:
                md = pd.read_csv(meta)[['Path', 'Age', 'AP/PA']].rename(columns={'AP/PA': 'APPA'})
                md['Path'] = md['Path'].astype(str)
                cl = cl.merge(md, on='Path', how='left')
            ours = []
            for df in _p2_split.values():
                for fname in df.loc[df['Source'] == 'CheXpert', 'Image Index']:
                    m = re.match(r'CheXpert_(patient\d+)_(\d+)_[A-Za-z]+_([A-Za-z]+)_(view\d+_frontal)', fname, re.I)
                    if m:
                        ours.append({'Image Index': fname, 'pid': m.group(1).lower(), 'age': int(m.group(2)),
                                     'view': m.group(3).upper(), 'viewfile': m.group(4)})
            ours = pd.DataFrame(ours)
            cand = ours.merge(cl[['Path', 'pid', 'viewfile'] + (['Age', 'APPA'] if have_meta else [])],
                              on=['pid', 'viewfile'], how='inner')
            if have_meta:
                ok = (cand['Age'].isna() | (cand['Age'] == cand['age'])) & \
                     (cand['APPA'].isna() | (cand['APPA'].astype(str).str.upper() == cand['view']))
                cand = cand[ok]
            n_c = cand.groupby('Image Index')['Path'].nunique()
            uniq = cand[cand['Image Index'].isin(n_c[n_c == 1].index)].drop_duplicates('Image Index')
            path_for = dict(zip(uniq['Image Index'], uniq['Path']))
            print(f"CheXpert label match: {len(path_for)} unique, {int((n_c > 1).sum())} ambiguous "
                  f"(ambiguous images keep partial labels)")
        lab = cl.set_index('Path')
        for fname, path in path_for.items():
            if path not in lab.index:
                continue
            r = lab.loc[path]
            r = r.iloc[0] if isinstance(r, pd.DataFrame) else r
            tgt = {c: 0 for c in P2_COMORB}
            for col, g in _CXP_TO_GROUP.items():
                if col in r.index and float(r[col]) == 1.0:
                    tgt[g] = 1
            tgt['Focal_Lesion_Chronic_Change'] = -1
            tgt['No_Finding'] = 1 if ('No Finding' in r.index and float(r['No Finding']) == 1.0) else 0
            out[fname] = tgt
    return out


_p2_ml = _p2_multilabel_lookup() if (P2_CFG['comorbidity_labels'] == 'partial' and not _P2_COHORT) else {}
for _k, _df in ({} if _P2_COHORT else _p2_split).items():
    rows = []
    for fname, grp in zip(_df['Image Index'], _df['Group']):
        if P2_CFG['comorbidity_labels'] == 'single':
            t = {c: int(c == grp) for c in P2_COMORB}
        else:
            t = p2_partial_targets(grp)
            if fname in _p2_ml:                       # true labels override "unknown", never contradict the folder
                for c, v in _p2_ml[fname].items():
                    if t[c] == -1:
                        t[c] = v
        rows.append([t[c] for c in P2_COMORB])
    _df[[f'com_{c}' for c in P2_COMORB]] = np.array(rows, dtype=np.int8)

_tr = _p2_split['train']
_known = (_tr[[f'com_{c}' for c in P2_COMORB]] >= 0).mean()
_copos = ((_tr['Pneumonia'] == 1) & (_tr['com_Cardiomegaly'] == 1)).sum()
print("Share of KNOWN comorbidity targets in training:\n", _known.round(3).to_string())
print(f"Training images labelled BOTH pneumonia and cardiomegaly: {_copos} "
      f"({'from multi-label files' if _copos else 'none: add the label CSVs in P1 to enable true co-occurrence'})")


## P3. Model

In [ ]:
# ============================================================================
# P3. MODEL: DenseNet-121 + spatial PAAG + bounded CSM (all variants share one class)
# ============================================================================
import torchvision


class SpatialPAAG(nn.Module):
    """Pathology-aware attention gate over the 1024 x h x w feature grid.
    a = sigmoid(conv1x1(ReLU(BN(conv1x1(F))))) in (0,1)^{h x w};  F_p = F * (1 + a)  (residual gating).
    The output bias starts at paag_init_bias (a ~ 0.05), so the gate begins close to identity and must
    learn to emphasise regions; it can never zero-out the global context."""

    def __init__(self, c=1024, hidden=256, init_bias=-3.0):
        super().__init__()
        self.reduce = nn.Conv2d(c, hidden, 1, bias=False)
        self.bn = nn.BatchNorm2d(hidden)
        self.score = nn.Conv2d(hidden, 1, 1)
        nn.init.zeros_(self.score.weight)
        nn.init.constant_(self.score.bias, init_bias)

    def forward(self, F_):
        a = torch.sigmoid(self.score(F.relu(self.bn(self.reduce(F_)))))
        return F_ * (1.0 + a), a


class BoundedCSM(nn.Module):
    """Comorbidity Suppressor: s = softplus(MLP(sigmoid(l_com))) >= 0, so it can only LOWER the pneumonia
    logit. With detach=True the comorbidity evidence is treated as fixed input: the pneumonia loss cannot
    reshape the comorbidity heads, which learn only from their own labels."""

    def __init__(self, n_com=5, hidden=32, init_bias=-3.0, detach=True):
        super().__init__()
        self.detach = detach
        self.net = nn.Sequential(nn.Linear(n_com, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        nn.init.zeros_(self.net[2].weight)
        nn.init.constant_(self.net[2].bias, init_bias)

    def forward(self, com_logits):
        x = torch.sigmoid(com_logits.detach() if self.detach else com_logits)
        return F.softplus(self.net(x))


class P2Net(nn.Module):
    def __init__(self, variant, pretrained=True):
        super().__init__()
        cfg = P2_VARIANTS[variant]
        self.variant = variant
        init = P2_CFG.get('backbone_init', 'imagenet')
        self.xrv_input = init.startswith('xrv:')
        if self.xrv_input:
            # torchxrayvision DenseNet-121 (1-channel input in [-1024, 1024]); weights trained on MIMIC-CXR or
            # PadChest only, so no CheXpert/NIH image (and hence no test image) was ever seen.
            try:
                import torchxrayvision as xrv
            except ImportError:
                import subprocess, sys
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchxrayvision'], check=True)
                import torchxrayvision as xrv
            os.makedirs(P2_PATHS['torchxrayvision_cache'], exist_ok=True)
            dn = xrv.models.DenseNet(weights=init[4:] if pretrained else None,
                                     cache_dir=P2_PATHS['torchxrayvision_cache'])
        else:
            try:
                dn = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
                                                    if pretrained else None)
            except Exception:
                dn = torchvision.models.densenet121(pretrained=pretrained)
        self.features = dn.features
        self.drop = nn.Dropout(P2_CFG.get('head_dropout', 0.0))
        self.paag = SpatialPAAG(1024, P2_CFG['paag_hidden'], P2_CFG['paag_init_bias']) if cfg['paag'] else None
        self.csm = BoundedCSM(len(P2_COMORB), P2_CFG['csm_hidden'], P2_CFG['csm_init_bias'],
                              P2_CFG['csm_detach']) if cfg['csm'] else None
        self.head_p = nn.Linear(1024, 1)
        self.head_c = nn.Linear(1024, len(P2_COMORB) + len(P2_AUX))   # 5 comorbidity groups (+ auxiliary findings)

    def forward(self, x):
        if self.xrv_input:              # undo ImageNet normalisation of the (identical) grey channels -> [-1024, 1024]
            x = (x[:, :1] * IMAGENET_STD[0] + IMAGENET_MEAN[0]) * 2048.0 - 1024.0
        Fm = F.relu(self.features(x))
        f_c = F.adaptive_avg_pool2d(Fm, 1).flatten(1)
        if self.paag is not None:
            Fp, a = self.paag(Fm)
            f_p = F.adaptive_avg_pool2d(Fp, 1).flatten(1)
        else:
            a, f_p = None, f_c
        p_raw = self.head_p(self.drop(f_p)).squeeze(1)
        com = self.head_c(self.drop(f_c))
        s = self.csm(com[:, :len(P2_COMORB)]).squeeze(1) if self.csm is not None else torch.zeros_like(p_raw)
        return {'p': p_raw - s, 'p_raw': p_raw, 'com': com, 'suppression': s, 'attn': a}

    def param_groups(self):
        back = list(self.features.parameters())
        head = [p for n, p in self.named_parameters() if not n.startswith('features.')]
        return back, head


class P2GradCAM:
    """Grad-CAM on the backbone output, for the raw pneumonia logit (default) or the CSM-corrected one."""

    def __init__(self, model):
        self.model = model
        self.acts = self.grads = None
        self.h1 = model.features.register_forward_hook(lambda m, i, o: setattr(self, 'acts', o))
        self.h2 = model.features.register_full_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0]))

    def __call__(self, x, target='p_raw'):
        self.model.zero_grad(set_to_none=True)
        out = self.model(x)
        out[target].sum().backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1)).detach().cpu().numpy()       # (B, h, w)
        cams = []
        for c in cam:
            c = (c - c.min()) / (c.max() - c.min() + 1e-8)
            cams.append(c)
        attn = out['attn'].detach().squeeze(1).cpu().numpy() if out['attn'] is not None else None
        return np.stack(cams), attn

    def close(self):
        self.h1.remove(); self.h2.remove()


def p2_count_params():
    rows = []
    for v in P2_VARIANTS:
        m = P2Net(v, pretrained=False)
        rows.append({'variant': v, 'params_M': sum(p.numel() for p in m.parameters()) / 1e6,
                     'added_vs_base_k': None})
    df = pd.DataFrame(rows)
    df['added_vs_base_k'] = (df['params_M'] - df.loc[df['variant'] == 'base', 'params_M'].iloc[0]) * 1e3
    return df


print(p2_count_params().round(3).to_string(index=False))


## P4. Loaders

In [ ]:
# ============================================================================
# P4. DATASETS, TRANSFORMS AND SEEDED LOADERS
# ============================================================================
from torchvision import transforms as T


def p2_transforms(train):
    S = P2_CFG['img_size']
    norm = [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    if not train:
        return T.Compose([T.Resize(P2_CFG['cache_short_side']), T.CenterCrop(S)] + norm)
    aug = [T.RandomResizedCrop(S, scale=P2_CFG['aug_scale'], ratio=P2_CFG['aug_ratio'], antialias=True),
           T.RandomAffine(degrees=P2_CFG['aug_degrees'],
                          translate=(P2_CFG['aug_translate'], P2_CFG['aug_translate'])),
           T.ColorJitter(brightness=P2_CFG['aug_brightness'], contrast=P2_CFG['aug_contrast'])]
    if P2_CFG['aug_hflip']:
        aug.append(T.RandomHorizontalFlip())
    return T.Compose(aug + norm)


class P2Dataset(Dataset):
    def __init__(self, df, train):
        self.df = df.reset_index(drop=True)
        self.tf = p2_transforms(train)
        self.y = self.df['Pneumonia'].astype(np.float32).values
        self.com = self.df[[f'com_{c}' for c in P2_COMORB] + [f'aux_{o}' for o in P2_AUX]].values.astype(np.float32)
        self.paths = self.df['Cache Path'].values

    def __len__(self):
        return len(self.df)

    def load_image(self, i):
        img = cv2.imread(self.paths[i], cv2.IMREAD_GRAYSCALE)
        return Image.fromarray(img).convert('RGB')

    def __getitem__(self, i):
        return self.tf(self.load_image(i)), torch.tensor(self.y[i]), torch.from_numpy(self.com[i]), i


def _p2_worker_init(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)


def p2_loader(split, train, seed=0, epoch=0, batch_size=None):
    ds = P2Dataset(_p2_split[split], train)
    g = torch.Generator(); g.manual_seed(int(seed) * 1000 + int(epoch))   # reproducible order + augmentation per epoch
    nw = P2_MACHINE['num_workers']
    return DataLoader(ds, batch_size=batch_size or P2_CFG['batch_size'], shuffle=train, drop_last=train,
                      num_workers=nw, pin_memory=torch.cuda.is_available(), generator=g,
                      worker_init_fn=_p2_worker_init, persistent_workers=False,
                      prefetch_factor=4 if nw > 0 else None)


def p2_pos_weights():
    tr = _p2_split['train']
    pos = (tr['Pneumonia'] == 1).sum(); neg = (tr['Pneumonia'] == 0).sum()      # masked (-1) images excluded
    w_p = min(neg / max(pos, 1), P2_CFG['max_pos_weight']) if P2_CFG['imbalance'] == 'pos_weight' else 1.0
    if P2_CFG.get('pos_weight_mode', 'full') == 'sqrt':
        w_p = float(np.sqrt(w_p))
    w_c = []
    for col_name in [f'com_{c}' for c in P2_COMORB] + [f'aux_{o}' for o in P2_AUX]:
        col = tr[col_name]
        kp, kn = (col == 1).sum(), (col == 0).sum()
        w_c.append(min(kn / max(kp, 1), P2_CFG['max_pos_weight']))
    return float(w_p), torch.tensor(w_c, dtype=torch.float32)


P2_POSW, P2_COMW = p2_pos_weights()
print(f"pneumonia pos_weight = {P2_POSW:.2f}; comorbidity pos_weights = {[round(x, 2) for x in P2_COMW.tolist()]}")


## P5. Training engine

In [ ]:
# ============================================================================
# P5. TRAINING ENGINE: EMA, warmup+cosine, AUC checkpointing, session time budget, resume, scoring
# ============================================================================
from sklearn.linear_model import LogisticRegression


def p2_run_dir(variant, seed, stage=None):
    stage = stage or P2_STAGE
    root = os.path.join(P2_PATHS['results_root'], f"{'dev_' if stage == 'dev' else ''}{P2_HASH}")
    return os.path.join(root, variant, f'seed_{seed}')


def p2_set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class EMA:
    """Exponential moving average of all floating-point state (weights and BN statistics).
    The effective decay warms up as min(decay, (1 + n) / (10 + n)) so early averages are not dominated by the
    randomly initialised heads (without this, a 0.999 EMA still carries ~47% initial weights after one epoch)."""

    def __init__(self, model, decay):
        self.decay = decay
        self.n = 0
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model):
        model.load_state_dict(self.shadow)


def _lr_lambda(total_steps, warmup_steps, min_frac):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_frac + (1 - min_frac) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    return f


def p2_loss(out, y, com, posw_p, posw_c):
    known = (y >= 0).float()                         # labeler-disagreement images (-1) do not train the pneumonia head
    lp = F.binary_cross_entropy_with_logits(out['p'], y.clamp(min=0), pos_weight=torch.tensor(posw_p, device=y.device),
                                            reduction='none')
    lp = (lp * known).sum() / known.sum().clamp(min=1.0)
    mask = (com >= 0).float()
    lc = F.binary_cross_entropy_with_logits(out['com'], com.clamp(min=0), pos_weight=posw_c.to(y.device),
                                            reduction='none')
    w = torch.ones(com.shape[1], device=y.device) * P2_CFG['lambda_com']
    w[len(P2_COMORB):] = P2_CFG.get('lambda_aux', 0.5)
    lc = (lc * mask * w).sum() / mask.sum().clamp(min=1.0)
    return lp + lc, lp.detach(), lc.detach()


@torch.no_grad()
def p2_predict(model, split, return_extra=False):
    model.eval()
    dl = p2_loader(split, train=False, batch_size=P2_CFG['batch_size'] * 2)
    P, PR, S, C = [], [], [], []
    for x, y, com, _ in dl:
        x = x.to(device, non_blocking=True)
        if P2_MACHINE['channels_last']:
            x = x.to(memory_format=torch.channels_last)
        with torch.autocast(device_type='cuda', enabled=P2_MACHINE['amp']):
            out = model(x)
        P.append(torch.sigmoid(out['p'].float()).cpu()); PR.append(torch.sigmoid(out['p_raw'].float()).cpu())
        S.append(out['suppression'].float().cpu()); C.append(torch.sigmoid(out['com'].float()).cpu())
    res = {'p': torch.cat(P).numpy(), 'p_raw': torch.cat(PR).numpy()}
    if return_extra:
        res.update({'suppression': torch.cat(S).numpy(), 'com': torch.cat(C).numpy()})
    return res


def _save_ckpt(path, **state):
    tmp = path + '.tmp'
    torch.save(state, tmp)
    os.replace(tmp, path)                          # atomic: a killed session never leaves a corrupt checkpoint


def p2_train(variant, seed, session_deadline=None):
    """Train (or resume) one (variant, seed). Returns 'complete', 'paused' or 'skipped'."""
    rd = p2_run_dir(variant, seed)
    os.makedirs(rd, exist_ok=True)
    if os.path.exists(os.path.join(rd, 'COMPLETE')):
        print(f"[{variant} | seed {seed}] already complete."); return 'skipped'
    last = os.path.join(rd, 'last.pth')
    p2_set_seed(seed)
    model = P2Net(variant).to(device)
    if P2_MACHINE['channels_last']:
        model = model.to(memory_format=torch.channels_last)
    back, head = model.param_groups()
    opt = torch.optim.AdamW([{'params': back, 'lr': P2_CFG['lr'] * P2_CFG['backbone_lr_mult']},
                             {'params': head, 'lr': P2_CFG['lr']}], weight_decay=P2_CFG['weight_decay'])
    steps_per_epoch = len(_p2_split['train']) // P2_CFG['batch_size']
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(steps_per_epoch * P2_CFG['epochs'],
                                                              steps_per_epoch * P2_CFG['warmup_epochs'],
                                                              P2_CFG['min_lr_frac']))
    scaler = (torch.amp.GradScaler('cuda', enabled=P2_MACHINE['amp']) if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler')
              else torch.cuda.amp.GradScaler(enabled=P2_MACHINE['amp']))
    ema = EMA(model, P2_CFG['ema_decay'])
    st = {'epoch': 0, 'best_auc': -1.0, 'best_epoch': 0, 'bad_epochs': 0, 'history': [], 'train_seconds': 0.0}
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        if ck['config_hash'] != P2_HASH:
            raise RuntimeError(f"{last} was made with config {ck['config_hash']}, current is {P2_HASH}. "
                               f"Refusing to mix configurations.")
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        scaler.load_state_dict(ck['scaler']); ema.shadow = ck['ema']; ema.n = ck.get('ema_n', 0); st = ck['state']
        random.setstate(ck['rng_py']); np.random.set_state(ck['rng_np']); torch.set_rng_state(ck['rng_torch'])
        if torch.cuda.is_available() and ck.get('rng_cuda') is not None:
            torch.cuda.set_rng_state_all(ck['rng_cuda'])
        print(f"[{variant} | seed {seed}] resuming after epoch {st['epoch']} (best val AUC {st['best_auc']:.4f} "
              f"at epoch {st['best_epoch']}).")
    else:
        save_run_manifest(rd, variant, seed, status='started')
    ema_model = P2Net(variant, pretrained=False).to(device)
    if P2_MACHINE['channels_last']:
        ema_model = ema_model.to(memory_format=torch.channels_last)
    yv = _p2_split['val']['Pneumonia'].astype(int).values
    kv = yv >= 0

    while st['epoch'] < P2_CFG['epochs'] and st['bad_epochs'] < P2_CFG['patience']:
        ep = st['epoch']
        avg_ep = st['train_seconds'] / ep if ep else None
        if session_deadline and avg_ep and time.time() + 1.15 * avg_ep > session_deadline:
            print(f"[{variant} | seed {seed}] stopping before epoch {ep + 1}: not enough session time left "
                  f"(~{avg_ep / 60:.0f} min/epoch). Progress is saved; re-run P7 next session to resume.")
            return 'paused'
        t0 = time.time()
        model.train()
        tot, totp, totc, n = 0.0, 0.0, 0.0, 0
        for x, y, com, _ in p2_loader('train', train=True, seed=seed, epoch=ep):
            x, y, com = x.to(device, non_blocking=True), y.to(device), com.to(device)
            if P2_MACHINE['channels_last']:
                x = x.to(memory_format=torch.channels_last)
            with torch.autocast(device_type='cuda', enabled=P2_MACHINE['amp']):
                out = model(x)
            out = {k: (v.float() if torch.is_tensor(v) else v) for k, v in out.items()}
            loss, lp, lc = p2_loss(out, y, com, P2_POSW, P2_COMW)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), P2_CFG['grad_clip'])
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            tot += loss.item() * len(y); totp += lp.item() * len(y); totc += lc.item() * len(y); n += len(y)
        ema.copy_to(ema_model)
        pv_ema = p2_predict(ema_model, 'val')['p']
        pv_raw = p2_predict(model, 'val')['p']
        auc_ema, auc_raw = roc_auc_score(yv[kv], pv_ema[kv]), roc_auc_score(yv[kv], pv_raw[kv])
        st['epoch'] = ep + 1
        st['train_seconds'] += time.time() - t0
        st['history'].append({'epoch': ep + 1, 'train_loss': tot / n, 'train_loss_pneu': totp / n,
                              'train_loss_com': totc / n, 'val_auc_ema': auc_ema, 'val_auc_raw': auc_raw,
                              'lr': opt.param_groups[1]['lr'], 'seconds': time.time() - t0})
        if auc_ema > st['best_auc'] + 1e-6:
            st['best_auc'], st['best_epoch'], st['bad_epochs'] = auc_ema, ep + 1, 0
            _save_ckpt(os.path.join(rd, 'best_ema.pth'), model=ema.shadow, epoch=ep + 1, val_auc=auc_ema,
                       config_hash=P2_HASH)
        else:
            st['bad_epochs'] += 1
        print(f"[{variant} | seed {seed}] epoch {ep + 1}/{P2_CFG['epochs']}  loss {tot / n:.4f}  "
              f"val AUC ema {auc_ema:.4f} (raw {auc_raw:.4f})  best {st['best_auc']:.4f}@{st['best_epoch']}  "
              f"{(time.time() - t0) / 60:.1f} min")
        _save_ckpt(last, model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                   scaler=scaler.state_dict(), ema=ema.shadow, ema_n=ema.n, state=st, config_hash=P2_HASH,
                   rng_py=random.getstate(), rng_np=np.random.get_state(), rng_torch=torch.get_rng_state(),
                   rng_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

    p2_finalize(variant, seed, st)
    return 'complete'


def p2_finalize(variant, seed, st):
    """Score the best EMA checkpoint on validation (and test in the confirm stage), fit Platt on validation."""
    rd = p2_run_dir(variant, seed)
    model = P2Net(variant, pretrained=False).to(device)
    model.load_state_dict(torch.load(os.path.join(rd, 'best_ema.pth'), map_location=device, weights_only=False)['model'])
    if P2_MACHINE['channels_last']:
        model = model.to(memory_format=torch.channels_last)
    val = p2_predict(model, 'val', return_extra=True)
    yv = _p2_split['val']['Pneumonia'].astype(int).values
    z = np.log(np.clip(val['p'], 1e-6, 1 - 1e-6) / np.clip(1 - val['p'], 1e-6, 1))
    kv = yv >= 0
    platt = LogisticRegression(C=1e6).fit(z[kv].reshape(-1, 1), yv[kv])
    arrays = {'val_index': np.asarray(_p2_split['val']['Image Index'], dtype=str), 'val_y': yv, 'val_p': val['p'],
              'val_p_raw': val['p_raw'], 'val_supp': val['suppression'], 'val_com': val['com']}
    if P2_STAGE == 'confirm':
        te = p2_predict(model, 'test', return_extra=True)
        arrays.update({'test_index': np.asarray(_p2_split['test']['Image Index'], dtype=str),
                       'test_y': _p2_split['test']['Pneumonia'].astype(int).values, 'test_p': te['p'],
                       'test_p_raw': te['p_raw'], 'test_supp': te['suppression'], 'test_com': te['com']})
        for extra in ('y_image', 'y_report', 'y_consensus'):
            if extra in _p2_split['test']:
                arrays[f'test_{extra}'] = _p2_split['test'][extra].astype(int).values
    np.savez_compressed(os.path.join(rd, 'predictions.npz'), **arrays)
    summary = {'variant': variant, 'seed': seed, 'config_hash': P2_HASH, 'stage': P2_STAGE,
               'best_epoch': st['best_epoch'], 'epochs_run': st['epoch'], 'best_val_auc_ema': st['best_auc'],
               'train_hours': st['train_seconds'] / 3600, 'platt_coef': float(platt.coef_[0, 0]),
               'platt_intercept': float(platt.intercept_[0]), 'history': st['history']}
    with open(os.path.join(rd, 'summary.json'), 'w') as fh:
        json.dump(summary, fh, indent=2, default=float)
    save_run_manifest(rd, variant, seed, status='complete')
    open(os.path.join(rd, 'COMPLETE'), 'w').write(time.strftime('%Y-%m-%d %H:%M:%S'))
    if os.path.exists(os.path.join(rd, 'last.pth')):
        os.remove(os.path.join(rd, 'last.pth'))       # keeps disk use down; best_ema.pth is the model of record
    print(f"[{variant} | seed {seed}] COMPLETE: best val AUC (EMA) {st['best_auc']:.4f} at epoch {st['best_epoch']}")


def save_run_manifest(rd, variant, seed, status):
    man = {'variant': variant, 'seed': seed, 'status': status, 'config_hash': P2_HASH, 'config': P2_CFG,
           'stage': P2_STAGE, 'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'torch': torch.__version__,
           'torchvision': torchvision.__version__, 'cuda': torch.version.cuda,
           'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}
    with open(os.path.join(rd, 'run_manifest.json'), 'w') as fh:
        json.dump(man, fh, indent=2, default=str)


## P6. Seed registry

In [ ]:
# ============================================================================
# P6. PRE-REGISTERED SEED REGISTRY AND PROGRESS BOARD
# ----------------------------------------------------------------------------
# The seed pool is generated ONCE from a fixed meta-seed and written to disk with a timestamp. Your five
# Protocol-1 seeds come first, so Protocol 1 and 2 can be compared seed for seed. You choose which seed to
# train each session, but which seeds "count" is fixed in advance: the confirmatory analysis uses the FIRST
# n_target_seeds of the registry and is released only when all of them are complete. Extra seeds beyond the
# target can be reported as a secondary analysis. This removes any room for picking favourable seeds.
# ============================================================================
P2_REGISTRY_PATH = os.path.join(P2_PATHS['results_root'], 'seed_registry.json')


def p2_seed_registry():
    if os.path.exists(P2_REGISTRY_PATH):
        with open(P2_REGISTRY_PATH) as fh:
            reg = json.load(fh)
    else:
        base = [42, 101, 2024, 7, 2025]
        rng = np.random.default_rng(20261015)
        extra = [int(s) for s in rng.choice(np.arange(10_000, 1_000_000), size=P2_CFG['seed_pool_size'] * 2,
                                            replace=False) if int(s) not in base]
        reg = {'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'meta_seed': 20261015,
               'n_target_seeds': P2_CFG['n_target_seeds'],
               'seeds': base + extra[:P2_CFG['seed_pool_size'] - len(base)]}
        with open(P2_REGISTRY_PATH, 'w') as fh:
            json.dump(reg, fh, indent=2)
        print(f"Seed registry created: {P2_REGISTRY_PATH}")
    return reg


def p2_status(variant, seed):
    rd = p2_run_dir(variant, seed)
    if os.path.exists(os.path.join(rd, 'COMPLETE')):
        return 'done'
    if os.path.exists(os.path.join(rd, 'last.pth')):
        try:
            ep = torch.load(os.path.join(rd, 'last.pth'), map_location='cpu', weights_only=False)['state']['epoch']
        except Exception:
            ep = '?'
        return f'ep{ep}'
    return '·'


def p2_board(show=True):
    reg = p2_seed_registry()
    rows = []
    for i, s in enumerate(reg['seeds']):
        r = {'#': i + 1, 'seed': s}
        for v in P2_VARIANTS:
            r[v] = p2_status(v, s)
        r['complete'] = all(r[v] == 'done' for v in P2_VARIANTS)
        rows.append(r)
    board = pd.DataFrame(rows)
    n_done = int(board['complete'].sum())
    if show:
        started = board[(board[list(P2_VARIANTS)] != '·').any(axis=1)]
        print(f"Protocol 2 [{P2_HASH}, {P2_STAGE}]: {n_done} complete seeds of target {reg['n_target_seeds']} "
              f"({len(started)} seeds started).")
        print((started if len(started) else board.head(5)).to_string(index=False))
        nxt = board[~board['complete']].head(3)
        print("Next seeds in registry order:", nxt['seed'].tolist())
    return board


_ = p2_board()


## P6b. Development sweep (validation only)

In [ ]:
# ============================================================================
# P6b. DEVELOPMENT SWEEP — choose the training recipe on VALIDATION only (the test set is never scored here)
# ----------------------------------------------------------------------------
# Your dev run showed the classic memorisation curve: training loss keeps falling (1.65 -> 0.61) while validation
# AUC peaks at epoch 3-4 and then drops. The network learns, then memorises the ~1,850 training positives, each
# amplified 14.7x by pos_weight. The sweep tests the two levers that address this, on the Base variant, seed 42:
#   * chest-X-ray pretraining that never saw CheXpert or ChestX-ray8 (MIMIC-CXR or PadChest DenseNet-121 from
#     torchxrayvision), so the features already encode radiographic findings and fewer epochs are needed;
#   * regularisation: lower LR, stronger weight decay, sqrt pos_weight, head dropout, shorter schedule.
# Each candidate is its own config hash and its own dev_ folder, and is resumable: if the session ends, run this
# cell again and it continues. Pick the best VALIDATION AUC, copy its settings into P2_CFG (P1), set
# P2_STAGE = 'confirm', set RUN_DEV_SWEEP = False, and start the pre-registered seeds in P7.
# Write the choice down before any confirm run: the recipe is then fixed for every seed and every variant.
# ============================================================================
RUN_DEV_SWEEP = False           # True -> run the candidates below now (P7 will then not train)
SWEEP_VARIANT, SWEEP_SEED = 'base', 42

_REG = {'lr': 1e-4, 'weight_decay': 0.05, 'pos_weight_mode': 'sqrt', 'head_dropout': 0.3, 'epochs': 12}
P2_SWEEP_CANDIDATES = {
    'current':            {},                                       # your dev run (reused if already complete)
    'regularised':        dict(_REG),
    'mimic':              {'backbone_init': 'xrv:densenet121-res224-mimic_ch'},
    'mimic_regularised':  {'backbone_init': 'xrv:densenet121-res224-mimic_ch', **_REG},
    # 'padchest':         {'backbone_init': 'xrv:densenet121-res224-pc'},
    # 'padchest_reg':     {'backbone_init': 'xrv:densenet121-res224-pc', **_REG},
}
_FORBIDDEN = ('-all', '-chex', '-nih', '-rsna', 'kaggle')    # these saw CheXpert / ChestX-ray8 images -> leakage


def p2_dev_sweep(candidates, variant='base', seed=42, hours=None):
    global P2_CFG, P2_STAGE, P2_HASH, P2_POSW, P2_COMW
    saved = (_copy.deepcopy(P2_CFG), P2_STAGE, P2_HASH, P2_POSW, P2_COMW)
    deadline = time.time() + (hours or P2_MACHINE['session_hours']) * 3600
    rows = []
    try:
        for name, over in candidates.items():
            bi = over.get('backbone_init', '')
            if any(t in bi for t in _FORBIDDEN):
                raise ValueError(f"{name}: {bi} was trained on CheXpert/ChestX-ray8 images (test leakage).")
            P2_CFG = _copy.deepcopy(saved[0]); P2_CFG.update(over)
            P2_STAGE = 'dev'
            P2_HASH = p2_config_hash(P2_CFG)
            P2_POSW, P2_COMW = p2_pos_weights()
            print(f"\n=== sweep candidate '{name}' | config {P2_HASH} | pos_weight {P2_POSW:.2f} | {over}")
            status = 'complete'
            if time.time() < deadline - 900:
                status = p2_train(variant, seed, session_deadline=deadline)
            rd = p2_run_dir(variant, seed, 'dev')
            row = {'candidate': name, 'config_hash': P2_HASH, **{k: over.get(k, saved[0].get(k)) for k in
                   ('backbone_init', 'lr', 'weight_decay', 'pos_weight_mode', 'head_dropout', 'epochs')}}
            sj = os.path.join(rd, 'summary.json')
            if os.path.exists(os.path.join(rd, 'COMPLETE')) and os.path.exists(sj):
                with open(sj) as fh:
                    s = json.load(fh)
                h = s.get('history', [])
                row.update({'best_val_auc': s['best_val_auc_ema'], 'best_epoch': s['best_epoch'],
                            'epochs_run': s['epochs_run'],
                            'last_val_auc': h[-1].get('val_auc_ema', np.nan) if h and isinstance(h[-1], dict) else np.nan})
            else:
                row['best_val_auc'] = np.nan; row['note'] = 'not finished - run this cell again'
            rows.append(row)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if status == 'paused':
                print("Session budget used; run this cell again in the next session to continue."); break
    finally:
        P2_CFG, P2_STAGE, P2_HASH, P2_POSW, P2_COMW = saved
    res = pd.DataFrame(rows).sort_values('best_val_auc', ascending=False)
    out = os.path.join(P2_PATHS['results_root'], 'dev_sweep.csv')
    res.to_csv(out, index=False)
    print("\nDev sweep (VALIDATION AUC only; the test set was not scored):")
    print(res.round(4).to_string(index=False))
    print(f"Saved to {out}. Differences under ~0.005 on one seed are within seed noise: prefer the simpler recipe.")
    return res


if RUN_DEV_SWEEP:
    _p2_sweep = p2_dev_sweep(P2_SWEEP_CANDIDATES, SWEEP_VARIANT, SWEEP_SEED)


## P7. Session runner (choose your seed here)

In [ ]:
# ============================================================================
# P7. SESSION RUNNER — choose the seed (and optionally the variants) for THIS session
# ----------------------------------------------------------------------------
# Set SESSION_SEED to a seed from the registry (or leave None to be asked / take the first incomplete one).
# SESSION_VARIANTS: list like ['base', 'paag'] or None = every unfinished variant of that seed, in order.
# The runner stops cleanly before the session time limit and resumes from the last finished epoch next time.
# Recommended: finish all four variants of a seed before starting the next, so the analysis stays balanced.
# ============================================================================
SESSION_SEED = None             # e.g. 42
SESSION_VARIANTS = None         # e.g. ['paag_csm']
SESSION_HOURS = P2_MACHINE['session_hours']
RUN_SESSION = True              # set False to only display the board

_reg = p2_seed_registry()
_board = p2_board(show=False)
_seed = SESSION_SEED or os.environ.get('P2_SEED')
if _seed in (None, ''):
    _default = int(_board.loc[~_board['complete'], 'seed'].iloc[0])
    try:
        _ans = input(f"Seed to train this session (registry order suggests {_default}; Enter = accept): ").strip()
    except Exception:
        _ans = ''
    _seed = int(_ans) if _ans else _default
_seed = int(_seed)
if _seed not in _reg['seeds']:
    raise ValueError(f"Seed {_seed} is not in the pre-registered pool ({P2_REGISTRY_PATH}). "
                     f"Pick one of: {_reg['seeds'][:10]} ...")
_pos = _reg['seeds'].index(_seed)
_skipped = [s for s in _reg['seeds'][:_pos] if not _board.loc[_board['seed'] == s, 'complete'].iloc[0]]
if _skipped:
    print(f"Note: earlier registry seeds {_skipped[:5]} are not complete yet. That is fine, but the confirmatory "
          f"analysis uses the FIRST {_reg['n_target_seeds']} registry seeds and waits until all of them are complete.")

_variants = SESSION_VARIANTS or [v for v in P2_VARIANTS if p2_status(v, _seed) != 'done']
_deadline = time.time() + SESSION_HOURS * 3600
print(f"Session plan: seed {_seed}, variants {_variants}, budget {SESSION_HOURS} h, config {P2_HASH} ({P2_STAGE})")

if RUN_SESSION and globals().get('RUN_DEV_SWEEP'):
    print("RUN_DEV_SWEEP is True in P6b, so this cell does not train. Set it to False once the configuration is frozen.")
elif RUN_SESSION:
    for _v in _variants:
        if time.time() > _deadline - 600:
            print("Session time nearly used; stopping before the next variant."); break
        _r = p2_train(_v, _seed, session_deadline=_deadline)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if _r == 'paused':
            break
_ = p2_board()


## P8. Pre-registered analysis

In [ ]:
# ============================================================================
# P8. PRE-REGISTERED ANALYSIS (CPU) — interim = estimates only; final = estimates + Holm-adjusted p-values
# ----------------------------------------------------------------------------
# Primary: test ROC-AUC differences PAAG-base, CSM-base, Full-base, estimated as the mean over seeds and
# tested with a hierarchical bootstrap that resamples SEEDS (training randomness) and PATIENTS (test sampling,
# longitudinal clustering) together. Secondary: all six pairs, within-CheXpert ROC-AUC, cardiomegaly
# false-positive rate at a validation-fixed specificity, recall at that threshold, Brier after Platt scaling,
# and the seed-level paired t-test.
# ============================================================================
from scipy.stats import rankdata

P2_OUT = os.path.join(P2_PATHS['results_root'], P2_HASH, 'analysis')
N_HBOOT = 2000
_hb_rng = np.random.default_rng(777)


def _auc(y, s):
    r = rankdata(s)
    n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def p2_collect(stage='confirm'):
    reg = p2_seed_registry()
    runs = {}
    for s in reg['seeds']:
        for v in P2_VARIANTS:
            rd = p2_run_dir(v, s, stage)
            f = os.path.join(rd, 'predictions.npz')
            if os.path.exists(os.path.join(rd, 'COMPLETE')) and os.path.exists(f):
                d = dict(np.load(f, allow_pickle=False))
                with open(os.path.join(rd, 'summary.json')) as fh:
                    d['summary'] = json.load(fh)
                if d['summary']['config_hash'] != P2_HASH:
                    continue
                runs[(v, s)] = d
    complete = [s for s in reg['seeds'] if all((v, s) in runs for v in P2_VARIANTS)]
    confirm_set = reg['seeds'][:reg['n_target_seeds']]
    final = all(s in complete for s in confirm_set)
    return runs, complete, confirm_set, final


def p2_run_metrics(d):
    te = _p2_split['test']
    assert np.array_equal(d['test_index'], np.asarray(te['Image Index'], dtype=str)), "test order changed"
    y, p = d['test_y'], d['test_p']
    k = y >= 0                                                                 # primary-label images only
    cxp = (te['Source'] == 'CheXpert').values
    yv, pv = d['val_y'], d['val_p']
    thr = float(np.quantile(pv[yv == 0], P2_CFG['fpr_spec_target']))           # val specificity = target
    pred = p >= thr
    card = ((te['Group'] == 'Cardiomegaly') & (te['Pneumonia'] == 0)).values
    a, b = d['summary']['platt_coef'], d['summary']['platt_intercept']
    z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1))
    pc = 1 / (1 + np.exp(-(a * z + b)))
    m = {'auc': _auc(y[k], p[k]), 'auc_chexpert': _auc(y[k & cxp], p[k & cxp]),
         'auc_raw_logit': _auc(y[k], d['test_p_raw'][k]), 'recall_at_spec': pred[y == 1].mean(),
         'spec_at_spec': 1 - pred[y == 0].mean(), 'cardio_fpr_at_spec': pred[card].mean(),
         'brier_raw': np.mean((p[k] - y[k]) ** 2), 'brier_platt': np.mean((pc[k] - y[k]) ** 2),
         'val_auc': _auc(yv[yv >= 0], pv[yv >= 0]), 'best_epoch': d['summary']['best_epoch']}
    if 'test_y_image' in d:                  # old (VisualCheXbert-only) definition on ALL test images: comparability
        m['auc_image_label_all'] = _auc(d['test_y_image'], p)
    return m


def _hier_boot(runs, seeds, a_, b_, mask=None):
    """Bootstrap distribution of mean_over_seeds(AUC_a - AUC_b), resampling seeds and patients jointly."""
    te = _p2_split['test']
    pid = te['Patient ID'].astype(str).values
    keep = (te['Pneumonia'].values >= 0) & (np.ones(len(te), bool) if mask is None else mask)
    codes, inv = np.unique(pid[keep], return_inverse=True)
    members = pd.Series(np.flatnonzero(keep)).groupby(inv).apply(np.array).values
    y_all = te['Pneumonia'].astype(int).values
    out = np.empty(N_HBOOT)
    for bi in range(N_HBOOT):
        ss = _hb_rng.choice(seeds, size=len(seeds), replace=True)
        idx = np.concatenate([members[k] for k in _hb_rng.integers(0, len(members), len(members))])
        y = y_all[idx]
        if y.min() == y.max():                       # single-class resample (only possible on tiny data)
            out[bi] = np.nan; continue
        cache = {}
        vals = []
        for s in ss:
            if s not in cache:
                cache[s] = _auc(y, runs[(a_, s)]['test_p'][idx]) - _auc(y, runs[(b_, s)]['test_p'][idx])
            vals.append(cache[s])
        out[bi] = np.mean(vals)
    return out[~np.isnan(out)]


def _holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * pvals[i]); adj[i] = min(1.0, run)
    return adj


def p2_analyze(stage='confirm', include_extra_seeds=False):
    os.makedirs(P2_OUT, exist_ok=True)
    runs, complete, confirm_set, final = p2_collect(stage)
    seeds = [s for s in complete if include_extra_seeds or s in confirm_set]
    status = 'FINAL' if final and not include_extra_seeds else ('SECONDARY (all complete seeds)' if include_extra_seeds
                                                                else 'INTERIM')
    print(f"{status}: {len(seeds)} complete seeds used (target set = first {len(confirm_set)} registry seeds; "
          f"{sum(s in complete for s in confirm_set)} of them complete).")
    if len(seeds) < 2:
        print("Need at least 2 complete seeds."); return None
    per = pd.DataFrame([{'variant': v, 'seed': s, **p2_run_metrics(runs[(v, s)])} for s in seeds for v in P2_VARIANTS])
    per.to_csv(os.path.join(P2_OUT, 'per_run_metrics.csv'), index=False)
    tab = per.groupby('variant').agg(['mean', 'std']).drop(columns='seed', level=0)
    print("\nPer-variant mean ± SD across seeds:")
    print(tab[[c for c in ['auc', 'auc_chexpert', 'auc_image_label_all', 'recall_at_spec', 'cardio_fpr_at_spec',
               'brier_platt', 'best_epoch'] if c in tab.columns.get_level_values(0)]]
          .reindex(list(P2_VARIANTS)).round(4).to_string())

    rows = []
    prim = [tuple(c) for c in P2_CFG['primary_contrasts']]
    pairs = prim + [p for p in [('paag', 'csm'), ('paag_csm', 'paag'), ('paag_csm', 'csm')] if p not in prim]
    cxp = (_p2_split['test']['Source'] == 'CheXpert').values
    for a_, b_ in pairs:
        d_seed = per.set_index(['variant', 'seed'])
        diffs = np.array([d_seed.loc[(a_, s), 'auc'] - d_seed.loc[(b_, s), 'auc'] for s in seeds])
        bd = _hier_boot(runs, seeds, a_, b_)
        row = {'contrast': f"{P2_VARIANTS[a_]['label']} - {P2_VARIANTS[b_]['label']}", 'primary': (a_, b_) in prim,
               'k_seeds': len(seeds), 'dAUC': diffs.mean(), 'ci_low': np.percentile(bd, 2.5),
               'ci_high': np.percentile(bd, 97.5), 'p_boot': min(1.0, 2 * min((bd <= 0).mean(), (bd >= 0).mean())),
               'p_seed_t': scipy_stats.ttest_1samp(diffs, 0).pvalue if diffs.std() > 0 else np.nan,
               'seeds_favouring_A': int((diffs > 0).sum())}
        for metric in ('auc_chexpert', 'cardio_fpr_at_spec', 'recall_at_spec', 'brier_platt'):
            dm = np.array([d_seed.loc[(a_, s), metric] - d_seed.loc[(b_, s), metric] for s in seeds])
            row[f'd_{metric}'] = dm.mean()
            row[f'p_seed_t_{metric}'] = scipy_stats.ttest_1samp(dm, 0).pvalue if dm.std() > 0 else np.nan
        if (a_, b_) in prim:
            bc = _hier_boot(runs, seeds, a_, b_, mask=cxp)
            row['dAUC_chexpert_ci'] = f"{np.percentile(bc, 2.5):+.4f} to {np.percentile(bc, 97.5):+.4f}"
        rows.append(row)
    res = pd.DataFrame(rows)
    res.loc[res['primary'], 'p_holm'] = _holm(res.loc[res['primary'], 'p_boot'].values)
    res.to_csv(os.path.join(P2_OUT, f"contrasts_{status.split()[0].lower()}.csv"), index=False)
    show = ['contrast', 'primary', 'k_seeds', 'dAUC', 'ci_low', 'ci_high', 'seeds_favouring_A', 'd_auc_chexpert',
            'd_cardio_fpr_at_spec', 'd_recall_at_spec', 'd_brier_platt']
    if status == 'FINAL' or include_extra_seeds:
        show += ['p_boot', 'p_holm', 'p_seed_t']
    else:
        print("\nINTERIM: estimates and 95% CIs only. Do not stop, add or drop seeds because of these numbers.")
    print("\nContrasts (A - B); dAUC CI from the seed x patient hierarchical bootstrap:")
    print(res[show].round(4).to_string(index=False))

    # forest plot of the primary contrasts
    fig, ax = plt.subplots(figsize=(7, 2.6))
    pr = res[res['primary']].reset_index(drop=True)
    ax.errorbar(pr['dAUC'], range(len(pr)), xerr=[pr['dAUC'] - pr['ci_low'], pr['ci_high'] - pr['dAUC']],
                fmt='o', color='#1f4e79', capsize=4)
    ax.axvline(0, color='grey', lw=1, ls='--')
    ax.set_yticks(range(len(pr))); ax.set_yticklabels(pr['contrast'])
    ax.set_xlabel('Δ test ROC-AUC (mean over seeds, 95% hierarchical-bootstrap CI)')
    ax.set_title(f'Protocol 2 primary contrasts — {status} ({len(seeds)} seeds)', fontsize=11)
    plt.tight_layout(); plt.savefig(os.path.join(P2_OUT, 'forest_primary.png'), dpi=300); plt.show()
    return per, res


_p2_res = p2_analyze()


## P9. Localization (first run; fell back to the central box)

In [ ]:
# ============================================================================
# P9. ANATOMICAL LOCALIZATION (Grad-CAM vs lung and heart masks)
# ----------------------------------------------------------------------------
# Reference masks come from the ChestX-Det PSPNet in torchxrayvision (Left/Right Lung, Heart), computed once on
# the exact network-input crop of each sampled image. If torchxrayvision or its weights are unavailable, the
# Protocol-1 central box is used as the 'lung' reference and heart metrics are skipped.
#   * pneumonia-positive images: share of CAM mass inside the lungs, heart excluded (higher = attends to lung
#     fields), and IoU
#     with the lung mask at the per-image optimal CAM threshold (comparable with Protocol 1);
#   * cardiomegaly images (pneumonia-negative): share of CAM mass on the heart (lower = less cardiac shortcut).
# For PAAG variants the gate's own attention map is scored the same way.
# Sample: a FIXED random sample (not 'the first N files') of pneumonia and cardiomegaly test images.
# ============================================================================
N_LOC_PNEU, N_LOC_CARD = 200, 200
LOC_DIR = os.path.join(P2_PATHS['results_root'], P2_HASH, 'localization')
os.makedirs(LOC_DIR, exist_ok=True)


def p2_loc_sample():
    f = os.path.join(LOC_DIR, 'sample.json')
    if os.path.exists(f):
        return json.load(open(f))
    te = _p2_split['test']
    rng = np.random.default_rng(31337)
    pneu = te.index[te['Pneumonia'] == 1].values
    card = te.index[(te['Group'] == 'Cardiomegaly') & (te['Pneumonia'] == 0)].values
    smp = {'pneumonia': sorted(rng.choice(pneu, min(N_LOC_PNEU, len(pneu)), replace=False).tolist()),
           'cardiomegaly': sorted(rng.choice(card, min(N_LOC_CARD, len(card)), replace=False).tolist())}
    json.dump(smp, open(f, 'w'))
    return smp


def p2_anatomy_masks(idx):
    f = os.path.join(LOC_DIR, 'anatomy_masks.npz')
    if os.path.exists(f):
        d = np.load(f)
        if np.array_equal(d['idx'], np.array(idx)):
            return d['lung'], d['heart'], str(d['source'])
    S = P2_CFG['img_size']
    ds = P2Dataset(_p2_split['test'], train=False)
    crop = T.Compose([T.Resize(P2_CFG['cache_short_side']), T.CenterCrop(S)])
    imgs = [np.array(crop(ds.load_image(i)).convert('L')) for i in idx]
    try:
        import torchxrayvision as xrv
        seg = xrv.baseline_models.chestx_det.PSPNet(cache_dir=P2_PATHS['torchxrayvision_cache']).to(device).eval()
        names = seg.targets
        il, ir, ih = names.index('Left Lung'), names.index('Right Lung'), names.index('Heart')
        lung, heart = [], []
        with torch.no_grad():
            for im in imgs:
                x = torch.from_numpy(xrv.datasets.normalize(im.astype(np.float32), 255)[None, None]).to(device)
                pr = torch.sigmoid(seg(x))[0].cpu().numpy()
                hm = (pr[ih] > 0.5).astype(np.uint8)
                # lung fields EXCLUDING the cardiac silhouette (the masks overlap behind the heart)
                lm = (((pr[il] > 0.5) | (pr[ir] > 0.5)) & (hm == 0)).astype(np.uint8)
                lung.append(cv2.resize(lm, (S, S), interpolation=cv2.INTER_NEAREST))
                heart.append(cv2.resize(hm, (S, S), interpolation=cv2.INTER_NEAREST))
        lung, heart, src = np.stack(lung), np.stack(heart), 'chestx_det_pspnet'
    except Exception as e:
        print(f"Anatomical segmentation unavailable ({type(e).__name__}: {e}); using the central box.")
        box = create_central_region_mask((S, S), center_ratio=0.5)
        lung, heart, src = np.repeat(box[None], len(idx), 0), np.zeros((len(idx), S, S), np.uint8), 'central_box'
    np.savez_compressed(f, idx=np.array(idx), lung=lung, heart=heart, source=src)
    return lung, heart, src


def _mass_in(cam, mask):
    return float((cam * mask).sum() / (cam.sum() + 1e-8))


def p2_localize_run(variant, seed, force=False):
    rd = p2_run_dir(variant, seed, 'confirm')
    out_f = os.path.join(rd, 'localization.json')
    if os.path.exists(out_f) and not force:
        return json.load(open(out_f))
    smp = p2_loc_sample()
    idx = smp['pneumonia'] + smp['cardiomegaly']
    lung, heart, src = p2_anatomy_masks(idx)
    S = P2_CFG['img_size']
    model = P2Net(variant, pretrained=False).to(device)
    model.load_state_dict(torch.load(os.path.join(rd, 'best_ema.pth'), map_location=device, weights_only=False)['model'])
    model.eval()
    gc = P2GradCAM(model)
    ds = P2Dataset(_p2_split['test'], train=False)
    rec = []
    for j, i in enumerate(idx):
        x = ds[i][0][None].to(device).requires_grad_(False)
        cam, attn = gc(x)
        cam = cv2.resize(cam[0], (S, S))
        r = {'i': int(i), 'kind': 'pneumonia' if j < len(smp['pneumonia']) else 'cardiomegaly',
             'cam_lung': _mass_in(cam, lung[j]), 'cam_heart': _mass_in(cam, heart[j]) if heart[j].any() else np.nan,
             'cam_iou_lung': compute_optimal_iou_threshold(cam, lung[j])[1]}
        if attn is not None:
            a = cv2.resize(attn[0], (S, S))
            r['attn_lung'] = _mass_in(a, lung[j])
            r['attn_heart'] = _mass_in(a, heart[j]) if heart[j].any() else np.nan
        rec.append(r)
    gc.close()
    df = pd.DataFrame(rec)
    summ = {'reference': src}
    for kind, cols in [('pneumonia', ['cam_lung', 'cam_iou_lung', 'attn_lung']),
                       ('cardiomegaly', ['cam_heart', 'attn_heart'])]:
        sub = df[df['kind'] == kind]
        for c in cols:
            if c in sub:
                summ[f'{kind}_{c}'] = float(sub[c].mean())
    json.dump(summ, open(out_f, 'w'), indent=2)
    return summ


def p2_localization_summary():
    runs, complete, confirm_set, final = p2_collect('confirm')
    rows = []
    for s in complete:
        for v in P2_VARIANTS:
            rows.append({'variant': v, 'seed': s, **p2_localize_run(v, s)})
    if not rows:
        print("No complete seeds yet."); return None
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(LOC_DIR, 'per_run_localization.csv'), index=False)
    metrics = [c for c in ['pneumonia_cam_lung', 'pneumonia_cam_iou_lung', 'cardiomegaly_cam_heart',
                           'pneumonia_attn_lung', 'cardiomegaly_attn_heart'] if c in df]
    print(f"Reference: {df['reference'].iloc[0]} | {df['seed'].nunique()} complete seeds")
    print(df.groupby('variant')[metrics].agg(['mean', 'std']).reindex(list(P2_VARIANTS)).round(4).to_string())
    for a_ in ['paag', 'csm', 'paag_csm']:
        for m in ['pneumonia_cam_lung', 'cardiomegaly_cam_heart']:
            if m not in df:
                continue
            piv = df.pivot(index='seed', columns='variant', values=m)
            d = (piv[a_] - piv['base']).dropna()
            line = f"  {P2_VARIANTS[a_]['label']:>10} - Base  {m:24s} mean {d.mean():+.4f} ({int((d > 0).sum())}/{len(d)} seeds > 0)"
            if final and len(d) > 2 and d.std() > 0:
                line += f"  seed-level t p = {scipy_stats.ttest_1samp(d, 0).pvalue:.3g}"
            print(line)
    return df


_p2_loc = p2_localization_summary()


## P9b. Anatomical localization (lung and heart masks)

In [ ]:
# ============================================================================
# P9b. ANATOMICAL LOCALISATION (replaces the central-box fallback used by P9)
# ----------------------------------------------------------------------------
# Run AFTER the shared cells and P1–P6 (no training needed; do NOT run P7; uses best_ema.pth of every complete confirmatory run).
# * Lung and heart masks: ChestX-Det PSPNet from torchxrayvision, run at its native 512 px on the exact 320 px
#   network-input crop of every sampled test image; lungs = (left | right lung) minus heart.
# * Same fixed sample as P9 (sample.json): 200 pneumonia-positive and 200 cardiomegaly (pneumonia-negative) images.
# * Per model: Grad-CAM mass inside lungs and best-threshold IoU with lungs (pneumonia images); Grad-CAM mass on the
#   heart (cardiomegaly images: lower = less cardiac shortcut); the same for PAAG's own attention map.
# * NO silent fallback: if the segmentation model cannot run, this cell stops with an error.
# * Results go to <results>/<hash>/localization_anatomical/ ; the old central-box results are kept untouched.
# Time: ~1 min for masks + ~20–40 s per model (80 models ≈ 30–50 min on an A6000). Re-running resumes.
# ============================================================================
import subprocess, sys
try:
    import torchxrayvision as xrv
except ImportError:                                   # the cause of the earlier fallback: package not installed
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchxrayvision'], check=True)
    import torchxrayvision as xrv

LOCA_DIR = os.path.join(P2_PATHS['results_root'], P2_HASH, 'localization_anatomical')
if 'p2_loc_sample' not in globals():                  # P9 not run in this session: reuse its saved sample
    def p2_loc_sample():
        return json.load(open(os.path.join(P2_PATHS['results_root'], P2_HASH, 'localization', 'sample.json')))
os.makedirs(LOCA_DIR, exist_ok=True)
os.makedirs(P2_PATHS['torchxrayvision_cache'], exist_ok=True)


def p2_anatomy_masks_strict(idx):
    """Lung (minus heart) and heart masks at network resolution for the sampled test images. Cached; raises on failure."""
    f = os.path.join(LOCA_DIR, 'anatomy_masks_pspnet.npz')
    if os.path.exists(f):
        d = np.load(f)
        if np.array_equal(d['idx'], np.array(idx)) and str(d['source']) == 'chestx_det_pspnet':
            return d['lung'], d['heart']
    S = P2_CFG['img_size']
    ds = P2Dataset(_p2_split['test'], train=False)
    crop = T.Compose([T.Resize(P2_CFG['cache_short_side']), T.CenterCrop(S)])     # identical to the model input crop
    seg = xrv.baseline_models.chestx_det.PSPNet(cache_dir=P2_PATHS['torchxrayvision_cache']).to(device).eval()
    il, ir, ih = seg.targets.index('Left Lung'), seg.targets.index('Right Lung'), seg.targets.index('Heart')
    lung, heart = [], []
    with torch.no_grad():
        for i in tqdm(idx, desc='segmenting'):
            im = np.array(crop(ds.load_image(i)).convert('L').resize((512, 512), Image.BILINEAR)).astype(np.float32)
            x = torch.from_numpy(xrv.datasets.normalize(im, 255)[None, None]).to(device)
            pr = torch.sigmoid(seg(x))[0].cpu().numpy()
            hm = (pr[ih] > 0.5).astype(np.uint8)
            lm = (((pr[il] > 0.5) | (pr[ir] > 0.5)) & (hm == 0)).astype(np.uint8)
            lung.append(cv2.resize(lm, (S, S), interpolation=cv2.INTER_NEAREST))
            heart.append(cv2.resize(hm, (S, S), interpolation=cv2.INTER_NEAREST))
    lung, heart = np.stack(lung), np.stack(heart)
    # quality control: masks must look anatomical, otherwise stop
    la, ha = lung.reshape(len(idx), -1).mean(1), heart.reshape(len(idx), -1).mean(1)
    print(f"Lung area fraction: median {np.median(la):.3f} (IQR {np.percentile(la, 25):.3f}–{np.percentile(la, 75):.3f}); "
          f"heart: median {np.median(ha):.3f}; empty lung masks: {(la < 0.02).sum()}, empty heart masks: {(ha < 0.005).sum()}")
    if np.median(la) < 0.10 or np.median(ha) < 0.02:
        raise RuntimeError("Segmentation masks look wrong (too small). Check the input crop before trusting the results.")
    np.savez_compressed(f, idx=np.array(idx), lung=lung, heart=heart, source='chestx_det_pspnet', lung_area=la, heart_area=ha)
    # QC figure: 8 random sampled images with their masks
    rng = np.random.default_rng(0); pick = rng.choice(len(idx), 8, replace=False)
    fig, axs = plt.subplots(2, 4, figsize=(12, 6))
    for ax, j in zip(axs.ravel(), pick):
        im = np.array(crop(ds.load_image(idx[j])).convert('L'))
        ax.imshow(im, cmap='gray'); ax.contour(lung[j], levels=[.5], colors='lime', linewidths=1)
        ax.contour(heart[j], levels=[.5], colors='red', linewidths=1); ax.axis('off')
    plt.suptitle('Segmentation QC: lungs minus heart (green), heart (red)'); plt.tight_layout()
    plt.savefig(os.path.join(LOCA_DIR, 'segmentation_qc.png'), dpi=150); plt.show()
    return lung, heart


def _mass(cam, m):
    return float((cam * m).sum() / (cam.sum() + 1e-8))


def p2_localize_anatomical(variant, seed, lung, heart, smp):
    out_f = os.path.join(LOCA_DIR, f'{variant}_seed{seed}.json')
    if os.path.exists(out_f):
        return json.load(open(out_f))
    rd = p2_run_dir(variant, seed, 'confirm')
    S = P2_CFG['img_size']
    model = P2Net(variant, pretrained=False).to(device)
    model.load_state_dict(torch.load(os.path.join(rd, 'best_ema.pth'), map_location=device, weights_only=False)['model'])
    model.eval()
    gc = P2GradCAM(model); ds = P2Dataset(_p2_split['test'], train=False)
    idx = smp['pneumonia'] + smp['cardiomegaly']; npn = len(smp['pneumonia']); rec = []
    for j, i in enumerate(idx):
        cam, attn = gc(ds[i][0][None].to(device))
        cam = cv2.resize(cam[0], (S, S))
        r = {'kind': 'pneumonia' if j < npn else 'cardiomegaly', 'cam_lung': _mass(cam, lung[j]),
             'cam_heart': _mass(cam, heart[j]), 'cam_iou_lung': compute_optimal_iou_threshold(cam, lung[j])[1]}
        if attn is not None:
            a = cv2.resize(attn[0], (S, S)); r['attn_lung'] = _mass(a, lung[j]); r['attn_heart'] = _mass(a, heart[j])
        rec.append(r)
    gc.close(); del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    df = pd.DataFrame(rec); summ = {'reference': 'chestx_det_pspnet'}
    for kind, cols in [('pneumonia', ['cam_lung', 'cam_iou_lung', 'cam_heart', 'attn_lung']),
                       ('cardiomegaly', ['cam_heart', 'cam_lung', 'attn_heart'])]:
        sub = df[df['kind'] == kind]
        for c in cols:
            if c in sub:
                summ[f'{kind}_{c}'] = float(sub[c].mean())
    json.dump(summ, open(out_f, 'w'), indent=2)
    return summ


def p2_localization_anatomical_summary():
    # complete confirmatory seeds (self-contained: does not need P8)
    reg = p2_seed_registry()
    complete = [sd for sd in reg['seeds'][:reg['n_target_seeds']]
                if all(os.path.exists(os.path.join(p2_run_dir(v, sd, 'confirm'), f)) for v in P2_VARIANTS
                       for f in ('COMPLETE', 'best_ema.pth'))]
    print(f"{len(complete)} complete confirmatory seeds found under {P2_PATHS['results_root']}/{P2_HASH}")
    if not complete:
        raise RuntimeError("No complete runs found: check that P1 printed config hash a9642dff9011 and stage = confirm.")
    smp = p2_loc_sample()                                     # same fixed sample as P9
    idx = smp['pneumonia'] + smp['cardiomegaly']
    lung, heart = p2_anatomy_masks_strict(idx)
    rows = []
    for s in tqdm(complete, desc='seeds'):
        for v in P2_VARIANTS:
            rows.append({'variant': v, 'seed': s, **p2_localize_anatomical(v, s, lung, heart, smp)})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(LOCA_DIR, 'per_run_localization_anatomical.csv'), index=False)
    metrics = [c for c in ['pneumonia_cam_lung', 'pneumonia_cam_iou_lung', 'pneumonia_cam_heart', 'cardiomegaly_cam_heart',
                           'cardiomegaly_cam_lung', 'pneumonia_attn_lung', 'cardiomegaly_attn_heart'] if c in df]
    print(f"Anatomical reference | {df['seed'].nunique()} seeds | {len(smp['pneumonia'])} pneumonia + "
          f"{len(smp['cardiomegaly'])} cardiomegaly images")
    print(df.groupby('variant')[metrics].agg(['mean', 'std']).reindex(list(P2_VARIANTS)).round(4).to_string())
    con = []
    for m in ['pneumonia_cam_lung', 'pneumonia_cam_iou_lung', 'cardiomegaly_cam_heart']:
        ps = []
        for a_ in ['paag', 'csm', 'paag_csm']:
            piv = df.pivot(index='seed', columns='variant', values=m); d = (piv[a_] - piv['base']).dropna()
            p = scipy_stats.ttest_1samp(d, 0).pvalue if d.std() > 0 else np.nan
            con.append({'metric': m, 'contrast': f"{P2_VARIANTS[a_]['label']} - Base", 'mean_diff': d.mean(), 'sd': d.std(),
                        'seeds_A_gt_B': int((d > 0).sum()), 'n': len(d), 'p_t': p}); ps.append(p)
        o = np.argsort(ps); adj = np.empty(3); run = 0                      # Holm within each metric
        for r_, k in enumerate(o):
            run = max(run, (3 - r_) * ps[k]); adj[k] = min(1, run)
        for k in range(3):
            con[-3 + k]['p_holm'] = adj[k]
    con = pd.DataFrame(con); con.to_csv(os.path.join(LOCA_DIR, 'contrasts_localization_anatomical.csv'), index=False)
    print("\nContrasts with Base (seed-level paired t, Holm within metric):")
    print(con.round(4).to_string(index=False))
    print(f"\nFiles to send back: {LOCA_DIR}/per_run_localization_anatomical.csv, "
          f"contrasts_localization_anatomical.csv, segmentation_qc.png")
    return df, con


_p2_loc_anat = p2_localization_anatomical_summary()


## P10. External evaluation on PadChest (keep PADCHEST_SHARE empty when sharing)

In [ ]:
# ============================================================================
# P10. EXTERNAL EVALUATION ON PADCHEST (no retraining: the 80 frozen confirmatory models are only scored)
# ----------------------------------------------------------------------------
# Cohort: PadChest images with PHYSICIAN-assigned labels, frontal (PA/AP), age >= 65  -> 12,038 images, 418 pneumonia,
# spread over 14 image archives (0–12 and 54). The manifest padchest_external_manifest.csv (made from the PadChest
# labels CSV) lists them with the pneumonia label and the five comorbidity groups the models were trained on.
# Step A streams one archive at a time: download -> extract ONLY the selected images -> convert 16-bit to 8-bit ->
#   the SAME CLAHE + resize as the training cache -> delete the raw image and the archive. Disk use stays small.
# Step B scores every complete (variant, seed) model on all images cached so far and saves one .npz per model.
# Both steps are resumable: re-run the cell any time; finished archives and models are skipped.
# Run after the shared cells and P1–P6 (do NOT run P7).
# ============================================================================
import zipfile, subprocess, sys

PADCHEST_DIR = os.path.join(BASE_DIR, 'padchest')
PADCHEST_MANIFEST = os.path.join(PADCHEST_DIR, 'padchest_external_manifest.csv')   # upload the file I sent here
PADCHEST_CACHE = os.path.join(PADCHEST_DIR, f"cache_clahe_{P2_CFG['cache_short_side']}")
PADCHEST_OUT = os.path.join(P2_PATHS['results_root'], P2_HASH, 'external_padchest')
KEEP_ARCHIVES = False            # True keeps each downloaded zip (needs ~tens of GB per archive)
RUN_DOWNLOAD, RUN_SCORING = True, True

# Archives to use, in order. The first four (3, 1, 8, 7) give 3,643 images with 154 pneumonia cases; see the
# justification in the manuscript. Add more archive numbers later if you want more precision (re-run resumes).
PADCHEST_ARCHIVES = [3, 1, 8, 7, 4, 54, 11, 0, 2, 9, 10, 5, 6, 12]   # all 14 hold selected images; most pneumonia first
# PRIVATE share link from the BIMCV access e-mail (do NOT publish it; blank it before uploading the notebook anywhere).
PADCHEST_SHARE = ''                       # private share link from your own BIMCV access e-mail (never publish it)
PADCHEST_SUBFOLDER = '/'                  # the zips sit at the top of the share
# 'remote'  : read ONLY the selected images straight out of each 20 GB zip over HTTP range requests (no zip on disk;
#             ~6–7 MB per image, i.e. ~6 GB per archive instead of 20 GB). Falls back to 'download' if the server refuses.
# 'download': download the whole zip (needs ~21 GB free disk per archive), extract, delete.
PADCHEST_MODE = 'remote'
REMOTE_THREADS = 8
_tok = PADCHEST_SHARE.rstrip('/').split('/')[-1] if PADCHEST_SHARE else ''
_host = PADCHEST_SHARE.split('/s/')[0] if PADCHEST_SHARE else ''
PADCHEST_URLS = {k: (f"{PADCHEST_SHARE}/download?path={PADCHEST_SUBFOLDER.replace('/', '%2F')}&files={k}.zip"
                     if PADCHEST_SHARE else '') for k in PADCHEST_ARCHIVES}
PADCHEST_WEBDAV = {k: (f"{_host}/public.php/webdav{PADCHEST_SUBFOLDER.rstrip('/')}/{k}.zip" if PADCHEST_SHARE else '')
                   for k in PADCHEST_ARCHIVES}
# Already have some zips on disk? Put them in PADCHEST_DIR/zips/ named 3.zip, 1.zip, ... and they are used directly.

for _d in (PADCHEST_DIR, PADCHEST_CACHE, PADCHEST_OUT, os.path.join(PADCHEST_DIR, 'zips')):
    os.makedirs(_d, exist_ok=True)
import ast, glob

def padchest_build_manifest(src, out):
    """Same selection and label mapping as padchest_filter.py."""
    d = pd.read_csv(src, low_memory=False)
    d['age'] = d['StudyDate_DICOM'] // 10000 - d['PatientBirth']
    d['lab'] = d['Labels'].map(lambda x: sorted({s.strip().lower() for s in ast.literal_eval(x) if isinstance(s, str) and s.strip()})
                               if isinstance(x, str) and x.startswith('[') else None)
    f = d[(d['MethodLabel'] == 'Physician') & d['Projection'].isin(['PA', 'AP', 'AP_horizontal']) & (d['age'] >= 65)
          & (d['Pediatric'] == 'No') & d['lab'].notna()].copy()

    has = lambda terms: f['lab'].map(lambda l: int(any(t in l for t in terms)))
    # primary reference: physician-assigned pneumonia diagnosis
    f['pneumonia'] = has(['pneumonia', 'atypical pneumonia'])
    # comorbidity groups, mapped to the five heads the models were trained with
    f['com_Cardiomegaly'] = has(['cardiomegaly'])
    f['com_Pleural_Extrapulmonary_Volume_Disease'] = has(['pleural effusion', 'costophrenic angle blunting', 'atelectasis',
        'laminar atelectasis', 'lobar atelectasis', 'segmental atelectasis', 'round atelectasis', 'atelectasis basal',
        'total atelectasis', 'pneumothorax', 'pleural thickening', 'apical pleural thickening', 'calcified pleural thickening',
        'hiatal hernia'])
    f['com_Weakly_Supervised_Uncertain'] = has(['pulmonary edema'])               # non-infectious airspace opacity (edema)
    f['com_Focal_Lesion_Chronic_Change'] = has(['emphysema', 'pulmonary fibrosis', 'nodule', 'multiple nodules',
                                                 'pulmonary mass', 'mass'])
    f['com_No_Finding'] = f['lab'].map(lambda l: int(l == ['normal']))
    f['heart_insufficiency'] = has(['heart insufficiency'])
    f['pleural_effusion'] = has(['pleural effusion'])
    # airspace findings without a pneumonia diagnosis: possible unlabelled infection (excluded in a sensitivity analysis)
    f['airspace_no_pneumonia'] = ((has(['consolidation', 'infiltrates', 'alveolar pattern', 'ground glass pattern']) == 1)
                                  & (f['pneumonia'] == 0)).astype(int)
    f['view'] = np.where(f['Projection'] == 'PA', 'PA', 'AP')
    cols = ['ImageID', 'ImageDir', 'PatientID', 'StudyID', 'StudyDate_DICOM', 'age', 'PatientSex_DICOM', 'view', 'Projection',
            'Modality_DICOM', 'Manufacturer_DICOM', 'BitsStored_DICOM', 'pneumonia', 'com_Cardiomegaly',
            'com_Focal_Lesion_Chronic_Change', 'com_No_Finding', 'com_Pleural_Extrapulmonary_Volume_Disease',
            'com_Weakly_Supervised_Uncertain', 'heart_insufficiency', 'pleural_effusion', 'airspace_no_pneumonia']
    m = f[cols].copy(); m['labels'] = f['lab'].map(lambda l: '|'.join(l))
    m.to_csv(out, index=False)
    print(f"Manifest built: {len(m)} images, {int(m.pneumonia.sum())} pneumonia -> {out}")


if not os.path.exists(PADCHEST_MANIFEST):
    _src = sorted(glob.glob(os.path.join(PADCHEST_DIR, 'PADCHEST_chest_x_ray_images_labels_160K*.csv*')) +
                  glob.glob(os.path.join(BASE_DIR, '**', 'PADCHEST_chest_x_ray_images_labels_160K*.csv*'), recursive=True))
    if not _src:
        raise FileNotFoundError(f"Upload either padchest_external_manifest.csv or the PadChest labels CSV "
                                f"(PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv.gz) to {PADCHEST_DIR}/")
    padchest_build_manifest(_src[0], PADCHEST_MANIFEST)
pc = pd.read_csv(PADCHEST_MANIFEST)
pc = pc[pc['ImageDir'].isin(PADCHEST_ARCHIVES)].reset_index(drop=True)
pc['Cache Path'] = [os.path.join(PADCHEST_CACHE, os.path.splitext(i)[0] + '.png') for i in pc['ImageID']]


def _pc_convert(raw_bytes, dst):
    """16-bit PadChest PNG -> 8-bit (0.5–99.5th percentile window) -> CLAHE -> short side 366, exactly as the training cache."""
    arr = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_UNCHANGED)
    if arr is None:
        return False
    if arr.ndim == 3:
        arr = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
    arr = arr.astype(np.float32)
    lo, hi = np.percentile(arr, [0.5, 99.5])
    img = np.clip((arr - lo) / max(hi - lo, 1e-6) * 255.0, 0, 255).astype(np.uint8)      # all selected images are MONOCHROME2
    img = cv2.createCLAHE(clipLimit=P2_CFG['clahe_clip'], tileGridSize=(P2_CFG['clahe_grid'], P2_CFG['clahe_grid'])).apply(img)
    h, w = img.shape
    s = P2_CFG['cache_short_side'] / min(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    cv2.imwrite(dst, img, [cv2.IMWRITE_PNG_COMPRESSION, 1])
    return True


def padchest_prepare_archive(k):
    done = os.path.join(PADCHEST_DIR, f'archive_{k}.done')
    if os.path.exists(done):
        return 'done'
    want = pc[pc['ImageDir'] == k]
    todo = {os.path.basename(i): p for i, p in zip(want['ImageID'], want['Cache Path']) if not os.path.exists(p)}
    if not todo:
        open(done, 'w').write('ok'); return 'done'
    zp = os.path.join(PADCHEST_DIR, 'zips', f'{k}.zip')
    if not os.path.exists(zp) or not zipfile.is_zipfile(zp):
        url = PADCHEST_URLS.get(k, '')
        if not url:
            print(f"archive {k}: no link and no zip on disk -> skipped"); return 'skipped'
        print(f"archive {k}: downloading ...", flush=True)
        r = subprocess.run(['wget', '-c', '--tries=50', '--timeout=60', '-q', '--show-progress', '-O', zp, url])
        if r.returncode != 0 or not zipfile.is_zipfile(zp):
            print(f"archive {k}: download incomplete (re-run the cell to resume)"); return 'partial'
    n_ok = 0
    with zipfile.ZipFile(zp) as z:
        members = {os.path.basename(m): m for m in z.namelist() if m.lower().endswith('.png')}
        for name, dst in tqdm(todo.items(), desc=f'archive {k}'):
            if name in members and _pc_convert(z.read(members[name]), dst):
                n_ok += 1
    missing = [n for n in todo if not os.path.exists(todo[n])]
    print(f"archive {k}: {n_ok} images cached, {len(missing)} not found in the zip")
    open(done, 'w').write(f'cached {n_ok}; missing {len(missing)}')
    if not KEEP_ARCHIVES:
        os.remove(zp)
    return 'done'


def _open_remote(k):
    from remotezip import RemoteZip
    last = None
    for url, kw in [(PADCHEST_WEBDAV[k], {'auth': (_tok, '')}), (PADCHEST_URLS[k], {})]:
        for suffix in (True, False):
            try:
                return RemoteZip(url, support_suffix_range=suffix, timeout=120, **kw)
            except Exception as e:                      # try the next endpoint / range style
                last = e
    raise RuntimeError(f"remote access to archive {k} failed: {last}")


def padchest_prepare_archive_remote(k):
    done = os.path.join(PADCHEST_DIR, f'archive_{k}.done')
    if os.path.exists(done):
        return 'done'
    want = pc[pc['ImageDir'] == k]
    todo = [(os.path.basename(i), p) for i, p in zip(want['ImageID'], want['Cache Path']) if not os.path.exists(p)]
    if not todo:
        open(done, 'w').write('ok'); return 'done'
    try:
        z0 = _open_remote(k)
    except Exception as e:
        print(f"archive {k}: {e}\n -> switching this archive to full download"); return padchest_prepare_archive(k)
    members = {os.path.basename(m): m for m in z0.namelist() if m.lower().endswith('.png')}
    z0.close()
    chunks = [todo[i::REMOTE_THREADS] for i in range(REMOTE_THREADS)]

    def work(chunk):
        n, z = 0, _open_remote(k)
        for name, dst in chunk:
            if name not in members:
                continue
            for attempt in range(5):
                try:
                    if _pc_convert(z.read(members[name]), dst):
                        n += 1
                    break
                except Exception:
                    time.sleep(5 * (attempt + 1)); z = _open_remote(k)
        return n
    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time()
    with ThreadPoolExecutor(REMOTE_THREADS) as ex:
        n_ok = sum(ex.map(work, chunks))
    missing = [n for n, p in todo if not os.path.exists(p)]
    print(f"archive {k}: {n_ok} images cached remotely in {(time.time() - t0) / 60:.1f} min; {len(missing)} missing "
          f"(re-run the cell to retry them)")
    if not missing:
        open(done, 'w').write(f'cached {n_ok} (remote)')
    return 'done' if not missing else 'partial'


class PadChestDS(Dataset):
    def __init__(self, paths):
        self.paths, self.tf = list(paths), p2_transforms(False)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = cv2.imread(self.paths[i], cv2.IMREAD_GRAYSCALE)
        return self.tf(Image.fromarray(img).convert('RGB')), i


def padchest_score_all():
    avail = pc[[os.path.exists(p) for p in pc['Cache Path']]].reset_index(drop=True)
    print(f"Scoring on {len(avail)} cached images ({int(avail['pneumonia'].sum())} pneumonia)")
    if avail.empty:
        return None
    reg = p2_seed_registry()
    seeds = [sd for sd in reg['seeds'][:reg['n_target_seeds']]
             if all(os.path.exists(os.path.join(p2_run_dir(v, sd, 'confirm'), 'best_ema.pth')) for v in P2_VARIANTS)]
    dl = DataLoader(PadChestDS(avail['Cache Path']), batch_size=P2_CFG['batch_size'] * 2, shuffle=False,
                    num_workers=P2_MACHINE['num_workers'], pin_memory=torch.cuda.is_available())
    ids = np.asarray(avail['ImageID'], dtype=str)
    for sd in tqdm(seeds, desc='seeds'):
        for v in P2_VARIANTS:
            f = os.path.join(PADCHEST_OUT, f'{v}_seed{sd}.npz')
            if os.path.exists(f) and np.array_equal(np.load(f)['image_id'], ids):
                continue
            model = P2Net(v, pretrained=False).to(device)
            model.load_state_dict(torch.load(os.path.join(p2_run_dir(v, sd, 'confirm'), 'best_ema.pth'),
                                             map_location=device, weights_only=False)['model'])
            model.eval()
            if P2_MACHINE['channels_last']:
                model = model.to(memory_format=torch.channels_last)
            P, PR, S, C = [], [], [], []
            with torch.no_grad():
                for x, _ in dl:
                    x = x.to(device, non_blocking=True)
                    if P2_MACHINE['channels_last']:
                        x = x.to(memory_format=torch.channels_last)
                    with torch.autocast(device_type='cuda', enabled=P2_MACHINE['amp']):
                        out = model(x)
                    P.append(torch.sigmoid(out['p'].float()).cpu()); PR.append(torch.sigmoid(out['p_raw'].float()).cpu())
                    S.append(out['suppression'].float().cpu()); C.append(torch.sigmoid(out['com'].float()).cpu())
            np.savez_compressed(f, image_id=ids, p=torch.cat(P).numpy(), p_raw=torch.cat(PR).numpy(),
                                supp=torch.cat(S).numpy(), com=torch.cat(C).numpy(),
                                com_names=np.array(list(P2_COMORB) + [f'aux_{o}' for o in P2_AUX]))
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    # quick look (full analysis is done separately): pneumonia ROC-AUC per variant
    y = avail['pneumonia'].values
    rows = []
    for v in P2_VARIANTS:
        for sd in seeds:
            d_ = np.load(os.path.join(PADCHEST_OUT, f'{v}_seed{sd}.npz'))
            rows.append({'variant': v, 'seed': sd, 'auc': roc_auc_score(y, d_['p'])})
    r = pd.DataFrame(rows)
    print(r.groupby('variant')['auc'].agg(['mean', 'std', 'min', 'max']).reindex(list(P2_VARIANTS)).round(4).to_string())
    print(f"\nSend back the folder {PADCHEST_OUT} (80 small .npz files) and {PADCHEST_MANIFEST}.")
    return r


if RUN_DOWNLOAD:
    try:
        import remotezip  # noqa
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip'], check=True)
    for _k in PADCHEST_ARCHIVES:
        local_zip = os.path.join(PADCHEST_DIR, 'zips', f'{_k}.zip')
        if PADCHEST_MODE == 'remote' and PADCHEST_SHARE and not os.path.exists(local_zip):
            padchest_prepare_archive_remote(_k)
        else:
            padchest_prepare_archive(_k)
if RUN_SCORING:
    _pc_res = padchest_score_all()


## P11. Exploratory follow-up: supervised CSM (Table S8)

In [ ]:
# ============================================================================
# P11. FOLLOW-UP: SUPERVISED CSM ("does the suppressor work when it is made to activate?")
# ----------------------------------------------------------------------------
# Add this cell at the END of the notebook. It changes nothing above it: same data, split, backbone, heads,
# schedule (6 epochs), optimiser, EMA, checkpoint rule and seeds. The ONLY difference from the confirmatory CSM
# variant is one extra loss term that trains the suppressor directly:
#
#   L_sup = lambda_sup * [ BCE(p_raw.detach() - s, 0) on comorbidity-positive, pneumonia-negative images
#                        + BCE(p_raw.detach() - s, 1) on pneumonia-positive images that also have a comorbidity ]
#
# p_raw is detached and the CSM input is already detached, so L_sup reaches ONLY the suppressor's own weights;
# the backbone and heads learn from exactly the same loss as before. The first group teaches "lower the score
# when a comorbidity explains it"; the second stops it from suppressing true pneumonia. Each group is averaged
# separately, so the term is balanced and needs no new pos_weight.
#
# Runs are written to results_P2/<new hash>/csm/seed_<s>/ (a new hash because the config gains one key);
# results_P2/a9642dff9011 is never touched. Comparison is paired with the existing Base and CSM runs of the
# same seeds.
#
# Steps (run the cell once per step; each step resumes if the session ends):
#   P11_STEP = 'pilot'    seed 42, VALIDATION only (dev stage, test never scored): checks the suppressor activates.
#                         Pre-declared rule: start at lambda_sup = 1; if mean suppression on comorbidity-positive,
#                         pneumonia-negative validation images is < 0.25 (inert CSM: 0.05), set
#                         P11_LAMBDA = 5 and run the pilot once more. No other value is tried.
#   P11_STEP = 'confirm'  the first 5 registry seeds with the chosen lambda, test scored as in P7.
#   P11_STEP = 'analyze'  paired comparison with Base and CSM (a9642dff9011) on the same 5 seeds.
# ============================================================================
import copy as _copy11

P11_STEP = 'pilot'            # 'pilot' -> 'confirm' -> 'analyze'
P11_LAMBDA = 1.0              # pre-declared: 1.0; only if the pilot is inert, 5.0 (once)
P11_N_SEEDS = 5               # first 5 registry seeds (paired with the existing Base and CSM runs)
P11_HOURS = P2_MACHINE['session_hours']
P11_ACTIVE_MIN = 0.25         # pre-declared activation threshold (logit units) for the pilot check
P11_COMORB_FOR_SUP = ['Cardiomegaly', 'Pleural_Extrapulmonary_Volume_Disease', 'Weakly_Supervised_Uncertain',
                      'Focal_Lesion_Chronic_Change']
_P11_BASE_HASH = 'a9642dff9011'

if '_p2_loss_original' not in globals():
    _p2_loss_original = p2_loss                    # keep the confirmatory loss, even if this cell is run again


def _p11_loss(out, y, com, posw_p, posw_c):
    loss, lp, lc = _p2_loss_original(out, y, com, posw_p, posw_c)
    if out.get('suppression') is None or not torch.is_tensor(out['suppression']):
        return loss, lp, lc
    idx = [P2_COMORB.index(c) for c in P11_COMORB_FOR_SUP]
    has_com = (com[:, idx] == 1).any(dim=1)
    neg = has_com & (y == 0)                       # comorbidity explains the image, no pneumonia
    pos = has_com & (y == 1)                       # true pneumonia alongside a comorbidity
    z = out['p_raw'].detach() - out['suppression']
    ls = z.new_zeros(())
    if neg.any():
        ls = ls + F.binary_cross_entropy_with_logits(z[neg], torch.zeros_like(z[neg]))
    if pos.any():
        ls = ls + F.binary_cross_entropy_with_logits(z[pos], torch.ones_like(z[pos]))
    lam = P2_CFG.get('csm_supervision', {}).get('lambda', 0.0)
    return loss + lam * ls, lp, lc


def _p11_enter(stage, lam):
    """Switch to the follow-up configuration; returns what is needed to switch back."""
    global P2_CFG, P2_STAGE, P2_HASH, P2_POSW, P2_COMW, p2_loss
    saved = (_copy11.deepcopy(P2_CFG), P2_STAGE, P2_HASH, P2_POSW, P2_COMW, p2_loss)
    if saved[2] != _P11_BASE_HASH:
        raise RuntimeError(f"P1 config hash is {saved[2]}, expected {_P11_BASE_HASH}: run P1 with the frozen "
                           "confirmatory configuration first.")
    P2_CFG = _copy11.deepcopy(saved[0])
    P2_CFG['csm_supervision'] = {'lambda': float(lam), 'groups': P11_COMORB_FOR_SUP,
                                 'target': 'p_raw.detach() - s', 'grad_to': 'suppressor only'}
    P2_STAGE = stage
    P2_HASH = p2_config_hash(P2_CFG)
    P2_POSW, P2_COMW = p2_pos_weights()
    p2_loss = _p11_loss
    return saved


def _p11_exit(saved):
    global P2_CFG, P2_STAGE, P2_HASH, P2_POSW, P2_COMW, p2_loss
    P2_CFG, P2_STAGE, P2_HASH, P2_POSW, P2_COMW, p2_loss = saved


def _p11_supp_stats(supp, df):
    y = df['Pneumonia'].astype(int).values
    com = np.stack([(df[f'com_{c}'] == 1).values for c in P11_COMORB_FOR_SUP], 1).any(1)
    card = (df['com_Cardiomegaly'] == 1).values
    return {'supp_all': float(supp.mean()), 'supp_comorbid_neg': float(supp[com & (y == 0)].mean()),
            'supp_cardio_neg': float(supp[card & (y == 0)].mean()), 'supp_pneu_pos': float(supp[y == 1].mean()),
            'supp_no_comorbid': float(supp[~com].mean())}


def _p11_write_protocol(lam, seeds):
    f = os.path.join(P2_PATHS['results_root'], P2_HASH, 'followup_protocol.json')
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f), exist_ok=True)
        json.dump({'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'purpose': 'supervised CSM follow-up (exploratory)',
                   'base_config': _P11_BASE_HASH, 'config_hash': P2_HASH, 'lambda': lam, 'seeds': seeds,
                   'comparators': ['base', 'csm'] , 'primary': 'activation: mean suppression on comorbidity-positive '
                   'pneumonia-negative test images vs inert CSM',
                   'secondary': ['cardiomegaly FPR at validation specificity 0.80 (paired vs Base and CSM)',
                                 'test ROC-AUC (paired vs Base and CSM)', 'recall at validation specificity 0.80'],
                   'note': 'pre-declared before any confirm-stage run; lambda fixed by the pilot rule on validation'},
                  open(f, 'w'), indent=1)
    return f


if P11_STEP == 'pilot':
    _s = _p11_enter('dev', P11_LAMBDA)
    try:
        print(f"P11 pilot | lambda {P11_LAMBDA} | config {P2_HASH} | seed 42 | validation only")
        _st = p2_train('csm', 42, session_deadline=time.time() + P11_HOURS * 3600)
        if _st != 'paused':
            _d = np.load(os.path.join(p2_run_dir('csm', 42), 'predictions.npz'))
            _stats = _p11_supp_stats(_d['val_supp'], _p2_split['val'])
            _ok = _stats['supp_comorbid_neg'] >= P11_ACTIVE_MIN
            print({k: round(v, 3) for k, v in _stats.items()})
            print(f"Activation {'REACHED' if _ok else 'NOT reached'} (needs >= {P11_ACTIVE_MIN}; inert CSM ~0.05). "
                  + ("Set P11_STEP = 'confirm' with this lambda." if _ok else
                     ("Set P11_LAMBDA = 5.0 and run the pilot once more." if P11_LAMBDA < 5 else
                      "Even lambda 5 is inert: report that the suppressor cannot be activated in this design; "
                      "you may still run 'confirm' to document it.")))
    finally:
        _p11_exit(_s)

elif P11_STEP == 'confirm':
    _reg = p2_seed_registry(); _seeds = _reg['seeds'][:P11_N_SEEDS]
    _s = _p11_enter('confirm', P11_LAMBDA)
    try:
        print(f"P11 confirm | lambda {P11_LAMBDA} | config {P2_HASH} | seeds {_seeds}")
        print(f"Protocol written to {_p11_write_protocol(P11_LAMBDA, _seeds)}")
        _deadline = time.time() + P11_HOURS * 3600
        for _sd in _seeds:
            if p2_train('csm', _sd, session_deadline=_deadline) == 'paused':
                print("Session time used; run this cell again next session to continue."); break
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    finally:
        _p11_exit(_s)

elif P11_STEP == 'analyze':
    from scipy import stats as _st11
    _reg = p2_seed_registry(); _seeds = _reg['seeds'][:P11_N_SEEDS]
    _s = _p11_enter('confirm', P11_LAMBDA); _new_hash = P2_HASH; _p11_exit(_s)
    _root = P2_PATHS['results_root']
    _arms = {'Base': (_P11_BASE_HASH, 'base'), 'CSM': (_P11_BASE_HASH, 'csm'), 'CSM-supervised': (_new_hash, 'csm')}
    _rows = []
    for _arm, (_h, _v) in _arms.items():
        for _sd in _seeds:
            _rd = os.path.join(_root, _h, _v, f'seed_{_sd}')
            if not os.path.exists(os.path.join(_rd, 'COMPLETE')):
                print(f"missing: {_arm} seed {_sd} ({_rd})"); continue
            _d = dict(np.load(os.path.join(_rd, 'predictions.npz')))
            _d['summary'] = json.load(open(os.path.join(_rd, 'summary.json')))
            _m = p2_run_metrics(_d)
            _m.update(_p11_supp_stats(_d['test_supp'], _p2_split['test']))
            _rows.append({'arm': _arm, 'seed': _sd, **_m})
    _res = pd.DataFrame(_rows)
    _cols = ['auc', 'cardio_fpr_at_spec', 'recall_at_spec', 'spec_at_spec', 'brier_platt', 'supp_comorbid_neg',
             'supp_cardio_neg', 'supp_pneu_pos', 'supp_no_comorbid', 'best_epoch']
    _out = os.path.join(_root, _new_hash, 'analysis'); os.makedirs(_out, exist_ok=True)
    _res.to_csv(os.path.join(_out, 'per_run_followup.csv'), index=False)
    print("Mean (SD) over seeds:")
    print(_res.groupby('arm')[_cols].agg(['mean', 'std']).round(4).T.to_string())
    _pair = []
    for _ref in ['Base', 'CSM']:
        for _c in ['auc', 'cardio_fpr_at_spec', 'recall_at_spec', 'supp_comorbid_neg']:
            _pv = _res.pivot(index='seed', columns='arm', values=_c)
            if 'CSM-supervised' not in _pv or _ref not in _pv:
                continue
            _dd = (_pv['CSM-supervised'] - _pv[_ref]).dropna()
            if len(_dd) >= 2:
                _pair.append({'contrast': f'CSM-supervised - {_ref}', 'metric': _c, 'n_seeds': len(_dd),
                              'mean_diff': _dd.mean(), 'sd': _dd.std(), 'n_positive': int((_dd > 0).sum()),
                              'p_paired_t': _st11.ttest_1samp(_dd, 0).pvalue})
    _pair = pd.DataFrame(_pair)
    _pair.to_csv(os.path.join(_out, 'paired_followup.csv'), index=False)
    print("\nPaired differences (same seeds):")
    print(_pair.round(4).to_string(index=False))
    print(f"\nSaved to {_out}. Send per_run_followup.csv and paired_followup.csv for the manuscript.")
